# CROCO — temporal validation (lean notebook)

## 1. Imports

In [ ]:
import sys, os, warnings, glob, gc, json, csv, time
# HDF5 file locking can make netCDF4 files on network/shared disks (e.g. /data)
# hang on open: disable it before netCDF/xarray are imported.
os.environ.setdefault('HDF5_USE_FILE_LOCKING', 'FALSE')
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

warnings.filterwarnings('ignore')

# Local xcroco package (CROCO python tools)
XCROCO_PATH = os.path.expanduser('./croco_pytools-v2.0.4/xcroco')
if XCROCO_PATH not in sys.path:
    sys.path.insert(0, XCROCO_PATH)

import inout as io      # I/O helpers to open CROCO grid + history/average files
import gridop as gop    # grid operations: staggering, rotation, vertical interpolation
from model import Model # native CROCO grid conventions

print('Setup complete')

## 2. Paths, flags and region boxes
Same files, rotation flag and lat/lon boxes as the training notebook.

In [ ]:
# ===== PATHS =====
DATA_DIR  = os.path.expanduser('./')
AVG_FILE  = os.path.join(DATA_DIR, 'croco_avg.0656.nc')   # training day (CROCO time-averaged output)
GRID_FILE = os.path.join(DATA_DIR, 'grd_alb500sm.nc')     # CROCO grid file (curvilinear, ~500 m)
T_IDX     = 0                                             # training snapshot within AVG_FILE
ROTATE    = False                                         # same flag as the training notebook (Section 4)

# ===== REGION BOXES (same lat/lon limits as the training notebook) =====
BOXES = {
    'Train':      (36.90, 38.30,  0.40,  2.00),   # (lat_min, lat_max, lon_min, lon_max)
    'Alboran':    (35.70, 36.40, -3.60, -2.20),
    'Transition': (36.40, 37.40, -1.50, -0.20),
}
print('Parameters defined')

## 3. Fixed z-levels and fields in the region boxes
`Z_LEVELS` from the deepest column at the training snapshot (as in the training notebook). `fields_at(path, t, regions)` reproduces Sections 3–7 of the training notebook (z of the sigma levels at that time, u,v on rho points, `isoslice` for u,v, column-by-column `interp1d` for w, k = 0 at the surface, crop and mask of the boxes), keeping in memory only the boxes. The training day is processed here for the three boxes: it provides the training data for the LR refit and the FiLM standardization.

In [ ]:
# ============================================================
# Fixed z-levels, region boxes and fields_at(): u, v, w on the fixed
# z-levels, computed ONLY inside the region boxes. Same operations as
# Sections 3-7 of the training notebook (get_z, to_rho, isoslice for u,v,
# column-by-column interp1d for w, flip k=0 surface, box crop and mask),
# without keeping any full-domain array in memory.
# ============================================================
croco = Model('croco_native')
_open = {}

def _open_file(path):
    if path not in _open:
        _open.clear()                                   # keep only the file in use open
        _open[path] = io.open_files(croco, GRID_FILE, [path], grid_metrics=1, chunks={'t': 1})
    return _open[path]

# --- Z_LEVELS: native sigma levels of the deepest column at the training snapshot (Section 3)
_ds1 = _open_file(AVG_FILE).isel(t=T_IDX)
_xg1 = gop.fast_xgcm_grid(_ds1, croco, grid_metrics=1)
_zr1 = gop.get_z(croco, ds=_ds1, z_sfc=_ds1.z_sfc, xgrid=_xg1, vgrid='r', hgrid='r')
_h = _ds1.h.values
j_deep, i_deep = np.unravel_index(np.nanargmax(_h), _h.shape)
Z_ASC = np.sort(np.asarray(_zr1[:, j_deep, i_deep].values))   # bottom -> surface
Z_LEVELS = Z_ASC[::-1].copy()                                # k = 0 surface (Section 6)
lon, lat = _ds1.lon.values, _ds1.lat.values
del _zr1, _xg1, _ds1
print(f'Z_LEVELS: {len(Z_LEVELS)} levels, {Z_LEVELS[0]:.1f} m (k=0) to {Z_LEVELS[-1]:.1f} m')

# --- region boxes: index ranges + mask of the points outside the lat/lon range (Section 7)
_BOX = {}
for reg, (la0, la1, lo0, lo1) in BOXES.items():
    m = (lat >= la0) & (lat <= la1) & (lon >= lo0) & (lon <= lo1)
    jj, ii = np.where(m)
    j0, j1, i0, i1 = jj.min(), jj.max(), ii.min(), ii.max()
    _BOX[reg] = dict(ys=slice(j0, j1 + 1), xs=slice(i0, i1 + 1), mask=~m[j0:j1 + 1, i0:i1 + 1],
                     lat=lat[j0:j1 + 1, i0:i1 + 1], lon=lon[j0:j1 + 1, i0:i1 + 1])
    print(f'{reg:<11} box {j1 - j0 + 1} x {i1 - i0 + 1}, valid points {m.sum()}')


VERBOSE = True   # print the time of every step of fields_at (to locate slow/hanging steps)

def _step(msg, t0):
    if VERBOSE:
        print(f'    [{time.time() - t0:6.1f} s] {msg}', flush=True)

def fields_at(path, t, regions):
    """u, v, w on Z_LEVELS (k=0 surface) inside the given region boxes, at
    time index t of `path`."""
    _ts = time.time()
    _step(f'open {os.path.basename(path)}, t={t}', _ts)
    d1 = _open_file(path).isel(t=t)
    xg = gop.fast_xgcm_grid(d1, croco, grid_metrics=1)
    zr = gop.get_z(croco, ds=d1, z_sfc=d1.z_sfc, xgrid=xg, vgrid='r', hgrid='r')
    zw = gop.get_z(croco, ds=d1, z_sfc=d1.z_sfc, xgrid=xg, vgrid='w', hgrid='r')
    _step('grid and z built (lazy)', _ts)
    if ROTATE:
        ur, vr = gop.rot_uv(d1.xcur.compute(), d1.ycur.compute(), d1.angle, xg)
        ug, vg = gop.to_rho(ur.compute(), xg), gop.to_rho(vr.compute(), xg)
    else:
        ug, vg = gop.to_rho(d1.xcur.compute(), xg), gop.to_rho(d1.ycur.compute(), xg)
    _step('u, v on rho points', _ts)
    zrc = zr.compute()
    _step('z_rho computed', _ts)
    uz = gop.isoslice(ug, Z_ASC, xg, target_data=zrc, axis='z').isel(z_r=slice(None, None, -1))
    vz = gop.isoslice(vg, Z_ASC, xg, target_data=zrc, axis='z').isel(z_r=slice(None, None, -1))
    _step('u, v isoslice set up', _ts)
    yd, xd = d1.zcur.dims[-2:]
    out = {}
    for reg in regions:
        b = _BOX[reg]
        U = uz.isel(y=b['ys'], x=b['xs']).compute().values
        V = vz.isel(y=b['ys'], x=b['xs']).compute().values
        zc  = d1.zcur.isel({yd: b['ys'], xd: b['xs']}).values
        zwb = zw.isel({zw.dims[-2]: b['ys'], zw.dims[-1]: b['xs']}).values
        W = np.full((len(Z_ASC),) + zc.shape[1:], np.nan)
        for jj in range(zc.shape[1]):
            for ii in range(zc.shape[2]):
                cz, cw = zwb[:, jj, ii], zc[:, jj, ii]
                ok = np.isfinite(cz) & np.isfinite(cw)
                if ok.sum() >= 2:
                    W[:, jj, ii] = interp1d(cz[ok], cw[ok], bounds_error=False, fill_value=np.nan)(Z_ASC)
        W = W[::-1]
        for A in (U, V, W):
            A[:, b['mask']] = np.nan
        out[reg] = (U, V, W)
        _step(f'{reg}: u, v, w in the box', _ts)
    del uz, vz, ug, vg, zr, zrc, zw, xg, d1
    gc.collect()
    return out


# --- training day, the three boxes (input of the FiLM setup cell below)
_t0 = time.time()
TRAIN_FIELDS = fields_at(AVG_FILE, T_IDX, ['Train', 'Alboran', 'Transition'])
U_IB, V_IB, W_IB = TRAIN_FIELDS['Train']
U_NA, V_NA, W_NA = TRAIN_FIELDS['Alboran']       # internal names as in the training notebook
U_AG, V_AG, W_AG = TRAIN_FIELDS['Transition']
lat_NA, lon_NA = _BOX['Alboran']['lat'], _BOX['Alboran']['lon']
lat_AG, lon_AG = _BOX['Transition']['lat'], _BOX['Transition']['lon']
print(f'Training day processed in {time.time() - _t0:.0f} s: '
      f'U_IB {U_IB.shape}, U_NA {U_NA.shape}, U_AG {U_AG.shape}')

## 4. FiLM — setup (no training)
Same setup cell as `notebook_nuredduna_seeds.ipynb`: parameters, model class, patch extraction, standardization from the training box.

In [ ]:
# ============================================================
# FiLM ENSEMBLE — SETUP (run once): parameters, model/helpers, patch
# extraction and standardization. Identical to the FiLM cell of
# notebook_nuredduna.ipynb; the seed-dependent part (training +
# evaluation) is in run_seed() below.
# ============================================================
import torch
import torch.nn as nn
import numpy as np
import time
import psutil
import os
import sys
import json
import csv
import itertools
from datetime import datetime
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.metrics import r2_score
from matplotlib.colors import SymLogNorm
# ============================================================
# Requires already loaded upstream (from the preprocessing block):
#   U_IB, V_IB, W_IB   -> (nz, ny, nx)  Ibiza fields (training)
#   U_NA, V_NA, W_NA   -> (nz, ny_na, nx_na)  Alboran fields (validation, REFERENCE)
#   lat_NA, lon_NA     -> 2D Alboran grids for plotting
#   U_AG, V_AG, W_AG   -> (nz, ny_ag, nx_ag)  Algerian fields (2nd validation)
#   lat_AG, lon_AG     -> 2D Algerian grids for plotting
#   Z_LEVELS           -> real depths (meters) per level
# ============================================================
import random
import subprocess

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ============================================================
# PARAMETERS — edit only here
# ============================================================
r_cnn          = 3         # HORIZONTAL stencil radius: patch = (2r+1) x (2r+1)
K_START_cnn    = 2         # level (index) of u,v used as input — e.g. surface/near-surface
N_TARGET_cnn   = 16        # levels 2..17 -> z down to about -116.6 m, stays above -150 m
K_PRED_cnn     = [2,7,12,17]   # absolute levels used for metrics/plots on Alboran and Algerian
ROW_BATCH_cnn   = 2048      # target number of (patch, level) ROWS per training batch: batches are
                            # built from whole patches, PATCH_BATCH_cnn = ROW_BATCH_cnn // N_TARGET_cnn
                            # patches with all their levels (conv1 once per patch, see forward_grouped)
PRED_BATCH_cnn  = 512       # batching for validation/prediction
K_EXTRAPOLATE_cnn = 20        # modest extrapolation just past training range (was 35)
N_EXTRAP_FIT_cnn  = 4         # deepest training levels used for the least-squares fit of mean_w(z)
                              # (linear) and log std_w(z) (linear) when predicting BELOW the training range
VALIDATE_EVERY_EPOCHS = 25      # cadence once training has settled
# Denser validation at the start, to resolve how val R2 rises: list of
# (up to epoch, check every N epochs); after the last entry the cadence
# is VALIDATE_EVERY_EPOCHS.
VALIDATE_SCHEDULE = [(25, 1)]

# Regularization knobs -- grouped here too (not just architecture/training
# params) so config.json is fully self-describing: no need to go read the
# model class / optimizer code to know how a run was
# regularized.
DROPOUT2D_P         = 0.1   # spatial dropout on conv feature maps (history: 0.1 -> 0.2 -> 0.1)
FC_DROPOUT_P        = 0.2   # dropout in the FC head (history: 0.3 -> 0.2)
WD_OTHER            = 1e-3  # weight_decay on conv1/conv2/FC weights (history: 1e-4 -> 1e-3 -> 3e-3 -> 1e-3)
WD_FILM             = 5e-5  # weight_decay (L2) on the FiLM generator only, see optimizer comment (history: 0.0 -> 1e-5 -> 5e-5 -> 1e-5 -> 5e-5)
# Stopping: NO fixed number of epochs. Training stops at the FIRST of
#  (a) the lr reaches LR_MIN_cnn: ReduceLROnPlateau (on the TRAIN loss) has
#      exhausted its reductions -> training has converged;
#  (b) the mean val R² on Alboran (K_PRED_cnn levels) has not improved for
#      R2_PATIENCE_EPOCHS_cnn epochs.
# Alboran only decides WHEN to stop: all evaluation uses the model at the
# stopping epoch (the final model), not the best-on-Alboran checkpoint.
LR_MIN_cnn             = 1e-6  # min lr of ReduceLROnPlateau = convergence criterion (a)
R2_PATIENCE_EPOCHS_cnn = 300   # criterion (b), in epochs (was 250 = 10 checks x 25)

# Save the run config alongside the results, so a results.npz/log.txt
# found later is self-describing without needing the notebook.
_config = dict(r_cnn=r_cnn, K_START_cnn=K_START_cnn, N_TARGET_cnn=N_TARGET_cnn,
                K_PRED_cnn=K_PRED_cnn,
                ROW_BATCH_cnn=ROW_BATCH_cnn, PRED_BATCH_cnn=PRED_BATCH_cnn,
                K_EXTRAPOLATE_cnn=K_EXTRAPOLATE_cnn, N_EXTRAP_FIT_cnn=N_EXTRAP_FIT_cnn,
                VALIDATE_EVERY_EPOCHS=VALIDATE_EVERY_EPOCHS,
                validate_schedule=VALIDATE_SCHEDULE,
                dropout2d_p=DROPOUT2D_P, fc_dropout_p=FC_DROPOUT_P,
                weight_decay_other=WD_OTHER, weight_decay_film=WD_FILM,
                lr_min=LR_MIN_cnn, r2_patience_epochs=R2_PATIENCE_EPOCHS_cnn,
                evaluated_model='final', device=str(device))
# (_config is written to config.json in every seed folder, together with the seed)


def extract_patches_with_all_levels(u_field, v_field, W_levels_full, r):
    """Extracts all (2r+1)x(2r+1) patches whose u,v neighbourhood is fully
    valid, together with the w vector at ALL target levels (even if some
    levels are NaN — they get filtered afterwards, row by row).
    Returns: patches (n_patches,2,ss,ss), y_all_levels (n_patches,n_levels)
             (can contain NaN), centers_j, centers_i (n_patches,) — indices
             in the original field.
    """
    ny, nx = u_field.shape
    ss = 2 * r + 1
    u_win = sliding_window_view(u_field, (ss, ss))
    v_win = sliding_window_view(v_field, (ss, ss))
    patch_valid = ~np.isnan(u_win).any(axis=(2, 3)) & ~np.isnan(v_win).any(axis=(2, 3))
    jj, ii = np.where(patch_valid)
    patches = np.stack([u_win[jj, ii], v_win[jj, ii]], axis=1)
    centers_j, centers_i = jj + r, ii + r
    y_all_levels = W_levels_full[:, centers_j, centers_i].T
    return patches, y_all_levels, centers_j, centers_i


def build_flat_training_rows(y_all_levels, z_of_level):
    """Expands (n_patches, n_levels) into independent (patch_idx, z, y) rows,
    dropping ONLY the (patch,level) combinations where w is NaN."""
    valid = ~np.isnan(y_all_levels)
    patch_idx, level_idx = np.where(valid)
    y_flat = y_all_levels[patch_idx, level_idx].astype(np.float32)
    z_flat = z_of_level[level_idx].astype(np.float32)
    return patch_idx, level_idx, z_flat, y_flat


class FiLM(nn.Module):
    """Generates gamma, beta from z and modulates a (B,C,H,W) tensor channel-wise:
    output = gamma * x + beta, applied identically over the whole spatial window."""
    def __init__(self, z_dim, num_channels):
        super().__init__()
        self.to_gamma_beta = nn.Linear(z_dim, num_channels * 2)
        # Canonical FiLM init: gamma starts at 1, beta at 0, i.e. FiLM starts
        # as the identity. This is only the INITIAL CONDITION: gamma/beta are
        # then free to move away from it during training. Without it,
        # nn.Linear's default weights give gamma~0 at the start and conv1's
        # features would be almost zeroed out before the network has learned
        # anything.
        nn.init.zeros_(self.to_gamma_beta.weight)
        nn.init.zeros_(self.to_gamma_beta.bias)
        with torch.no_grad():
            self.to_gamma_beta.bias[:num_channels].fill_(1.0)  # half of gamma -> 1, beta stays 0

    def forward(self, x, z):
        gamma_beta = self.to_gamma_beta(z)
        gamma, beta = gamma_beta.chunk(2, dim=1)
        gamma = gamma.unsqueeze(-1).unsqueeze(-1)
        beta  = beta.unsqueeze(-1).unsqueeze(-1)
        return gamma * x + beta


class ZCondStencilCNN_FiLM(nn.Module):
    """CNN conditioned on z via a single FiLM layer, applied BEFORE
    conv2 (it modulates conv1's output). conv1 sees only the raw (u,v)
    stencil (no z at all) and always extracts the same depth-invariant
    local features; FiLM then rescales/shifts those 8 feature maps as a
    function of z, and conv2 -- a fixed, non-z-dependent 8->16
    recombination -- operates on that already depth-conditioned input.
    Single entry point for z, ~7.8k total parameters."""
    def __init__(self, stencil_size, z_dim=1):
        super().__init__()
        self.conv1 = nn.Conv2d(2, 8, kernel_size=3, padding=1)   # padding=1 -- same-size output (7x7 -> 7x7)
        self.film1 = FiLM(z_dim, 8)                               # only entry point for z -- applied BEFORE conv2
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=0)   # no FiLM: fixed recombination of the z-modulated input
        self.relu = nn.ReLU()
        self.spatial_dropout = nn.Dropout2d(DROPOUT2D_P)  # value set in the PARAMETERS block above
        flat_dim = 16 * (stencil_size - 2) * (stencil_size - 2)   # conv1 is now 'same' padding (no shrink); only conv2 is valid (-2)
        self.fc = nn.Sequential(
            nn.Linear(flat_dim, 16), nn.ReLU(), nn.Dropout(FC_DROPOUT_P),
            nn.Linear(16, 1)
        )

    def features(self, x_img):
        """Depth-invariant part (conv1 + ReLU): depends only on the (u,v) stencil."""
        return self.relu(self.conv1(x_img))

    def head(self, h1, z):
        """z-dependent part. Dropout on conv1's output is applied HERE, i.e.
        per (patch, z) row, so every row keeps its own independent mask
        exactly as before the conv1 hoisting."""
        x = self.spatial_dropout(h1)
        x = self.film1(x, z)                                      # z enters here, BEFORE conv2
        x = self.spatial_dropout(self.relu(self.conv2(x)))        # fixed recombination of the z-modulated input
        feat = x.flatten(start_dim=1)
        return self.fc(feat).squeeze(-1)

    def forward(self, x_img, z):
        return self.head(self.features(x_img), z)

    def forward_grouped(self, x_unique, inv, z):
        """Same as forward(x_unique[inv], z), but conv1 runs once per distinct
        patch; its output is then copied to every row that shares the patch."""
        return self.head(self.features(x_unique)[inv], z)


def predict_zcond_in_batches(model, patches_tensor, patch_idx, z_flat_s, device,
                              batch_size=PRED_BATCH_cnn):
    """Mini-batch inference. Within each chunk, conv1 is computed once per
    distinct patch (forward_grouped) and shared by all rows using it."""
    model.eval()
    n = len(patch_idx)
    z_t = torch.tensor(z_flat_s, dtype=torch.float32).unsqueeze(1)
    out = []
    with torch.no_grad():
        for start in range(0, n, batch_size):
            end = start + batch_size
            uniq_b, inv_b = np.unique(patch_idx[start:end], return_inverse=True)
            xb = patches_tensor[uniq_b].to(device)
            inv_t = torch.as_tensor(inv_b.ravel(), dtype=torch.long, device=device)
            zb = z_t[start:end].to(device)
            out.append(model.forward_grouped(xb, inv_t, zb).cpu())
    return torch.cat(out, dim=0).numpy()


# ------------------------------------------------------------
# 1. Extraction — Ibiza (training): horizontal patches + all w levels
# ------------------------------------------------------------
u_cnn = U_IB[K_START_cnn]
v_cnn = V_IB[K_START_cnn]
W_IB_slice = W_IB[K_START_cnn : K_START_cnn + N_TARGET_cnn]
z_levels_ib = Z_LEVELS[K_START_cnn : K_START_cnn + N_TARGET_cnn]
patches_ib, y_all_ib, _, _ = extract_patches_with_all_levels(u_cnn, v_cnn, W_IB_slice, r_cnn)
patch_idx_ib, level_idx_ib, z_flat_ib, y_flat_ib = build_flat_training_rows(y_all_ib, z_levels_ib)
stencil_size_cnn = 2 * r_cnn + 1
print(f'Ibiza — stencil {stencil_size_cnn}×{stencil_size_cnn} | valid patches: {len(patches_ib)} | '
      f'rows (patch,level) for training: {len(y_flat_ib)}')
if len(y_flat_ib) == 0:
    raise RuntimeError('No valid training rows — check r_cnn/K_START_cnn')

# ------------------------------------------------------------
# 2. Standardization
# ------------------------------------------------------------
mean_u_cnn, std_u_cnn = patches_ib[:, 0].mean(), patches_ib[:, 0].std()
mean_v_cnn, std_v_cnn = patches_ib[:, 1].mean(), patches_ib[:, 1].std()
patches_ib_s = patches_ib.copy().astype(np.float32)
patches_ib_s[:, 0] = (patches_ib[:, 0] - mean_u_cnn) / std_u_cnn
patches_ib_s[:, 1] = (patches_ib[:, 1] - mean_v_cnn) / std_v_cnn
mean_z_cnn, std_z_cnn = z_levels_ib.mean(), z_levels_ib.std()
z_flat_ib_s = (z_flat_ib - mean_z_cnn) / std_z_cnn
mean_w_per_level = np.array([np.nanmean(y_all_ib[:, t]) for t in range(N_TARGET_cnn)])
std_w_per_level  = np.array([np.nanstd(y_all_ib[:, t])  for t in range(N_TARGET_cnn)]) + 1e-10
y_flat_ib_s = (y_flat_ib - mean_w_per_level[level_idx_ib]) / std_w_per_level[level_idx_ib]
patches_ib_tensor = torch.tensor(patches_ib_s, dtype=torch.float32)

# Target stats (mean_w, std_w) at an ARBITRARY depth z [m]. They are only
# defined on the training levels, so:
#  - z between two training levels -> linear interpolation in z between the
#    two nearest levels (weights by distance); exact value on a level;
#  - z below the deepest training level -> least-squares fit on the
#    N_EXTRAP_FIT_cnn deepest levels, evaluated at z: mean_w(z) linear,
#    log(std_w)(z) linear and then std_w = exp(...), so std_w stays > 0;
#  - z above the shallowest training level -> not supported (error).
_z_train = np.asarray(z_levels_ib, dtype=np.float64)
_asc = np.argsort(_z_train)                     # ascending z: deepest first

def w_stats_at_z(z):
    z = float(z)
    z_top, z_bot = _z_train.max(), _z_train.min()
    if z > z_top + 1e-6:
        raise ValueError(f'z={z:.2f} m is above the shallowest training level ({z_top:.2f} m): not supported')
    if z >= z_bot - 1e-6:                        # inside the training range
        m = np.interp(z, _z_train[_asc], mean_w_per_level[_asc])
        s = np.interp(z, _z_train[_asc], std_w_per_level[_asc])
    else:                                        # below the training range
        sel = _asc[:N_EXTRAP_FIT_cnn]            # the N deepest training levels
        m = np.polyval(np.polyfit(_z_train[sel], mean_w_per_level[sel], 1), z)
        # std_w: linear fit of log(std_w) in z, then back to std_w with exp (always > 0)
        s = np.exp(np.polyval(np.polyfit(_z_train[sel], np.log(std_w_per_level[sel]), 1), z))
    return float(m), float(s)

# ------------------------------------------------------------
# 3. Extraction — Alboran (validation)
# ------------------------------------------------------------
u_na, v_na = U_NA[K_START_cnn], V_NA[K_START_cnn]
ny_na, nx_na = u_na.shape
W_NA_slice = W_NA[K_START_cnn : K_START_cnn + N_TARGET_cnn]
z_levels_na = Z_LEVELS[K_START_cnn : K_START_cnn + N_TARGET_cnn]
patches_na, y_all_na, centers_j_na, centers_i_na = extract_patches_with_all_levels(u_na, v_na, W_NA_slice, r_cnn)
patches_na_s = patches_na.copy().astype(np.float32)
patches_na_s[:, 0] = (patches_na[:, 0] - mean_u_cnn) / std_u_cnn
patches_na_s[:, 1] = (patches_na[:, 1] - mean_v_cnn) / std_v_cnn
patches_na_tensor = torch.tensor(patches_na_s, dtype=torch.float32)
print(f'Alboran — valid patches: {len(patches_na)}')

# ------------------------------------------------------------
# 3b. Extraction — Algerian (2nd independent validation zone)
# ------------------------------------------------------------
u_ag, v_ag = U_AG[K_START_cnn], V_AG[K_START_cnn]
ny_ag, nx_ag = u_ag.shape
W_AG_slice = W_AG[K_START_cnn : K_START_cnn + N_TARGET_cnn]
patches_ag, y_all_ag, centers_j_ag, centers_i_ag = extract_patches_with_all_levels(u_ag, v_ag, W_AG_slice, r_cnn)
patches_ag_s = patches_ag.copy().astype(np.float32)
patches_ag_s[:, 0] = (patches_ag[:, 0] - mean_u_cnn) / std_u_cnn
patches_ag_s[:, 1] = (patches_ag[:, 1] - mean_v_cnn) / std_v_cnn
patches_ag_tensor = torch.tensor(patches_ag_s, dtype=torch.float32)
print(f'Algerian — valid patches: {len(patches_ag)}')

## 4b. Fast reader for the validation snapshots
`fields_at_np` computes the same fields as `fields_at` with numpy, reading from the file only the region boxes (z of the sigma levels with the CROCO Vtransform=2 formula, u,v averaged onto rho points, linear interpolation onto `Z_LEVELS`). It is checked here against `fields_at` at the training snapshot, at the 16 target levels, and then used for all the other snapshots: xcroco/xgcm process the whole domain and on `/data` take minutes per snapshot and exhaust the memory.

In [ ]:
# ============================================================
# Fast reader for the validation snapshots: the same fields as fields_at()
# (Sections 3-7 of the training notebook), computed with numpy reading from
# the file ONLY the region boxes (the files are chunked, so this is fast and
# needs little memory; xcroco/xgcm instead load and interpolate the whole
# domain, which on /data takes minutes per snapshot and exhausts the memory).
#   z of the sigma levels: CROCO Vtransform=2 (same formula as gop.get_z),
#       z = zeta + (zeta + h) (hc sc + h Cs) / (hc + h)
#   u, v on rho points: mean of the two neighbouring u (v) points (same as
#       gop.to_rho, i.e. xgcm interp, away from the domain edges)
#   vertical interpolation onto Z_LEVELS: linear, NaN outside the column
#       (as isoslice for u, v and interp1d for w)
# It is checked below against fields_at() at the training snapshot.
# ============================================================
import netCDF4

with netCDF4.Dataset(GRID_FILE) as _g:
    _H = np.asarray(_g['h'][:], dtype=np.float64)
    _lat_nc = np.asarray(_g['lat_rho'][:], dtype=np.float64)
    _lon_nc = np.asarray(_g['lon_rho'][:], dtype=np.float64)

# xcroco drops the outer ring of rho points, so its (y, x) indices -- used by
# _BOX -- are shifted with respect to the raw NetCDF ones. Find the offset by
# matching the xcroco lat/lon (cell 3) inside the NetCDF grid.
_OFF = None
for _oj in range(_lat_nc.shape[0] - lat.shape[0] + 1):
    for _oi in range(_lat_nc.shape[1] - lat.shape[1] + 1):
        if (np.allclose(_lat_nc[_oj:_oj + lat.shape[0], _oi:_oi + lat.shape[1]], lat)
                and np.allclose(_lon_nc[_oj:_oj + lat.shape[0], _oi:_oi + lat.shape[1]], lon)):
            _OFF = (_oj, _oi)
            break
    if _OFF is not None:
        break
assert _OFF is not None, 'cannot align the xcroco grid with the NetCDF grid'
print(f'xcroco grid {lat.shape} = NetCDF grid {_lat_nc.shape} shifted by (j, i) = {_OFF}')


def _interp_columns(z, f, z_new):
    """Linear interpolation of every column f(z) (axis 0) onto z_new; NaN outside."""
    nz, ny, nx = f.shape
    out = np.full((len(z_new), ny, nx), np.nan)
    for jj in range(ny):
        for ii in range(nx):
            cz, cf = z[:, jj, ii], f[:, jj, ii]
            ok = np.isfinite(cz) & np.isfinite(cf)
            if ok.sum() >= 2:
                out[:, jj, ii] = np.interp(z_new, cz[ok], cf[ok], left=np.nan, right=np.nan)
    return out


def fields_at_np(path, t, regions):
    """Same output as fields_at(path, t, regions), reading only the boxes."""
    out = {}
    with netCDF4.Dataset(path) as d:
        hc = float(d.hc)
        sc_r, Cs_r = np.asarray(d.sc_r, float), np.asarray(d.Cs_r, float)
        sc_w, Cs_w = np.asarray(d.sc_w, float), np.asarray(d.Cs_w, float)
        for reg in regions:
            b = _BOX[reg]
            j0, j1 = b['ys'].start + _OFF[0], b['ys'].stop + _OFF[0]     # NetCDF indices
            i0, i1 = b['xs'].start + _OFF[1], b['xs'].stop + _OFF[1]
            assert j0 >= 1 and i0 >= 1, 'box touches the domain edge'
            zeta = np.asarray(d['zeta'][t, j0:j1, i0:i1], dtype=np.float64)
            h = _H[j0:j1, i0:i1]
            u = np.asarray(d['u'][t, :, j0:j1, i0 - 1:i1], dtype=np.float64)
            v = np.asarray(d['v'][t, :, j0 - 1:j1, i0:i1], dtype=np.float64)
            w = np.asarray(d['w'][t, :, j0:j1, i0:i1], dtype=np.float64)
            ur = 0.5 * (u[:, :, :-1] + u[:, :, 1:])
            vr = 0.5 * (v[:, :-1, :] + v[:, 1:, :])
            zr = zeta + (zeta + h) * (hc * sc_r[:, None, None] + h * Cs_r[:, None, None]) / (hc + h)
            zw = zeta + (zeta + h) * (hc * sc_w[:, None, None] + h * Cs_w[:, None, None]) / (hc + h)
            U = _interp_columns(zr, ur, Z_ASC)[::-1]
            V = _interp_columns(zr, vr, Z_ASC)[::-1]
            W = _interp_columns(zw, w, Z_ASC)[::-1]
            for A in (U, V, W):
                A[:, b['mask']] = np.nan
            out[reg] = (U, V, W)
    return out


# --- check against fields_at() (xcroco) at the training snapshot
_t0 = time.time()
_NP = fields_at_np(AVG_FILE, T_IDX, ['Train', 'Alboran', 'Transition'])
print(f'numpy reader: training day in {time.time() - _t0:.1f} s')
_ok_all = True
for reg in ['Train', 'Alboran', 'Transition']:
    for name, A, B in zip('UVW', _NP[reg], TRAIN_FIELDS[reg]):
        k = slice(K_START_cnn, K_START_cnn + N_TARGET_cnn)          # levels used by the models
        a, bb = A[k], B[k]
        nan_diff = int((np.isnan(a) != np.isnan(bb)).sum())
        both = np.isfinite(a) & np.isfinite(bb)
        scale = np.nanmax(np.abs(bb))
        rel = float(np.max(np.abs(a[both] - bb[both])) / scale) if both.any() else 0.0
        ok = nan_diff == 0 and rel < 1e-4
        _ok_all &= ok
        print(f'  {reg:<11} {name}: max |diff|/max|field| = {rel:.1e}, NaN mismatches = {nan_diff}'
              f'{"" if ok else "   <-- DIFFERENT"}')
if not _ok_all:
    raise RuntimeError('the numpy reader does not reproduce fields_at() at the training snapshot')
print('  ✓ numpy reader reproduces fields_at() at the training snapshot')
del _NP

## 5. Temporal validation
Models trained on the training day (LR refitted, deterministic; FiLM = `model_final.pt` of seed `SEED_FILM`) are evaluated, without retraining, on Alboran and Transition at other days: one snapshot (time index 0) of one raw file every `FILE_EVERY` in `RAW_DIR`. First the training snapshot is checked against the stored R² (`lr_reference.json`, `result.json`): it must be reproduced. Output in `temporal_validation/`: one CSV per snapshot (resumable), `temporal_r2_all.csv`, `temporal_snapshots.csv` (time of every snapshot: model time, days since initialization, calendar date if `MODEL_START_DATE` is set, days from training), tables and `temporal_r2.png/pdf`.

In [ ]:
# ============================================================
# TEMPORAL VALIDATION — models trained on the training day, validated at
# other days. Nothing is retrained:
#   - per-level LR refitted on the training box at the training snapshot
#     (deterministic, identical to Section 9 of the training notebook);
#   - FiLM = final weights (model_final.pt) of ONE trained seed.
# For every snapshot the fields are processed with fields_at() (above) and
# both models are evaluated on the validation regions against the true w.
# ============================================================
import torch

# ============================================================
# PARAMETERS — edit only here
# ============================================================
# Each raw file holds one day (8 three-hourly averages); the file number
# grows by 8 per day, so the lag from the training day is (N - 656)/8 days.
RAW_DIR      = '/data/geo/lamarca/ROMS500/raw'   # all CROCO avg files on the server
FILE_EVERY   = 7                                 # one raw file every 7 (about one per week), time index 0
REGIONS      = ['Alboran', 'Transition']         # validation regions
SEED_FILM    = 261                               # representative ensemble member (paper Fig. 10)
ENSEMBLE_DIR = './cnn_seeds_film'
TEMPORAL_DIR = './temporal_validation'
PRED_BATCH_T = 16 * PRED_BATCH_cnn
# Calendar date of the model initialization (CROCO stores 'time' as
# "averaged time since initialization" in seconds, without a reference
# date). Set it, e.g. '2016-09-01 00:00', to also save the calendar date of
# every snapshot; leave None to save only the model time.
MODEL_START_DATE = None
# ============================================================

assert r_cnn == 3 and K_START_cnn == 2 and N_TARGET_cnn == 16
os.makedirs(TEMPORAL_DIR, exist_ok=True)
_raw = sorted(glob.glob(os.path.join(RAW_DIR, 'croco_avg.*.nc')))
TIME_SOURCES = [(AVG_FILE, T_IDX)] + [                 # training snapshot, then one snapshot per selected day
    (f, 0) for f in _raw[::FILE_EVERY]
    if os.path.basename(f) != os.path.basename(AVG_FILE)]
print(f'{len(_raw)} raw files found in {RAW_DIR}; {len(TIME_SOURCES)} snapshots to evaluate')


def _time_seconds(path, t):
    with xr.open_dataset(path, decode_times=False) as d:
        tv = 'time' if 'time' in d.variables else 'ocean_time'
        return float(d[tv].values[t])


def _date_of(sec):
    if MODEL_START_DATE is None:
        return ''
    import pandas as pd
    return (pd.Timestamp(MODEL_START_DATE) + pd.Timedelta(seconds=sec)).strftime('%Y-%m-%d %H:%M')


_T0 = _time_seconds(AVG_FILE, T_IDX)

# ------------------------------------------------------------
# Models (no training)
# ------------------------------------------------------------
def lr_features(p):
    n = len(p)
    return np.concatenate([p[:, 0].reshape(n, -1), p[:, 1].reshape(n, -1)], axis=1)

_X_ib = lr_features(patches_ib)
lr_models = {}
for _t in range(N_TARGET_cnn):
    _ok = ~np.isnan(y_all_ib[:, _t])
    lr_models[_t] = LinearRegression().fit(_X_ib[_ok], y_all_ib[_ok, _t])

_seed_dir = os.path.join(ENSEMBLE_DIR, f'seed_{SEED_FILM:03d}')
film = ZCondStencilCNN_FiLM(stencil_size_cnn).to(device)
film.load_state_dict(torch.load(os.path.join(_seed_dir, 'model_final.pt'), map_location=device))
film.eval()
print(f'LR refitted on the training box ({len(lr_models)} levels); FiLM = seed {SEED_FILM} (model_final.pt)')

K_ALL = list(range(K_START_cnn, K_START_cnn + N_TARGET_cnn))
_z_tr = Z_LEVELS[K_START_cnn:K_START_cnn + N_TARGET_cnn]


def evaluate(U, V, W):
    """R² of LR and FiLM at all 16 target levels for one region and one time."""
    Ws = W[K_START_cnn:K_START_cnn + N_TARGET_cnn]
    p, y_all, _, _ = extract_patches_with_all_levels(U[K_START_cnn], V[K_START_cnn], Ws, r_cnn)
    X = lr_features(p)
    r_lr = {}
    for t in range(N_TARGET_cnn):
        ok = ~np.isnan(y_all[:, t])
        r_lr[t] = float(r2_score(y_all[ok, t], lr_models[t].predict(X[ok])))
    ps = p.astype(np.float32).copy()
    ps[:, 0] = (p[:, 0] - mean_u_cnn) / std_u_cnn
    ps[:, 1] = (p[:, 1] - mean_v_cnn) / std_v_cnn
    pidx, lidx, zf, yf = build_flat_training_rows(y_all, _z_tr)
    ys = predict_zcond_in_batches(film, torch.tensor(ps), pidx, (zf - mean_z_cnn) / std_z_cnn,
                                  device, batch_size=PRED_BATCH_T)
    yp = ys * std_w_per_level[lidx] + mean_w_per_level[lidx]
    r_film = {t: float(r2_score(yf[lidx == t], yp[lidx == t])) for t in range(N_TARGET_cnn)}
    return r_lr, r_film, len(p)


# ------------------------------------------------------------
# Check at the training snapshot: R² must reproduce the stored values
# (LR: lr_reference.json; FiLM: result.json of the seed). If the lean
# processing differed from the training notebook, these would not match.
# ------------------------------------------------------------
_ref_film = json.load(open(os.path.join(_seed_dir, 'result.json')))['r2_final']
_lr_ref_path = os.path.join(ENSEMBLE_DIR, 'lr_reference.json')
_ref_lr = json.load(open(_lr_ref_path)).get('r2', {}) if os.path.exists(_lr_ref_path) else {}
_KEY = {'Alboran': 'Alboran', 'Transition': 'Algerian'}      # internal keys of the stored files
print('\nCheck at the training snapshot (computed vs stored):')
_bad = 0
for reg in REGIONS:
    r_lr, r_film, _ = evaluate(*TRAIN_FIELDS[reg])
    for k in K_PRED_cnn:
        t_ = k - K_START_cnn
        f_ref = _ref_film.get(f'{_KEY[reg]}|{k}', _ref_film.get(f'{reg}|{k}'))
        l_ref = (_ref_lr.get(_KEY[reg]) or _ref_lr.get(reg) or {}).get(str(k), {}).get('r2')
        ok_f = f_ref is None or abs(r_film[t_] - f_ref) < 1e-3
        ok_l = l_ref is None or abs(r_lr[t_] - l_ref) < 1e-3
        _bad += (not ok_f) + (not ok_l)
        print(f'  {reg:<11} k={k:<3} FiLM {r_film[t_]:.4f} (stored {f_ref if f_ref is None else round(f_ref, 4)}) '
              f'| LR {r_lr[t_]:.4f} (stored {l_ref if l_ref is None else round(l_ref, 4)})'
              f'{"" if ok_f and ok_l else "   <-- MISMATCH"}')
if _bad:
    raise RuntimeError('training-snapshot R² does not reproduce the stored values: check the processing')
print('  ✓ training snapshot reproduced')

# ------------------------------------------------------------
# Loop over time snapshots (resumable: one CSV per snapshot)
# ------------------------------------------------------------
_FIELDS = ['file', 't_idx', 'time_s', 'days_since_init', 'date', 'hours',
           'region', 'model', 'k', 'z_m', 'r2', 'n_patches']
for (path, t) in TIME_SOURCES:
    tag = f'{os.path.splitext(os.path.basename(path))[0]}_t{t:02d}'
    out_csv = os.path.join(TEMPORAL_DIR, f'r2_{tag}.csv')
    if os.path.exists(out_csv):
        print(f'{tag}: already done, skipped'); continue
    t0 = time.time()
    tsec = _time_seconds(path, t)
    hours = (tsec - _T0) / 3600.0
    tinfo = [tsec, tsec / 86400.0, _date_of(tsec), hours]
    F = TRAIN_FIELDS if (path, t) == (AVG_FILE, T_IDX) else fields_at_np(path, t, REGIONS)
    rows = []
    for reg in REGIONS:
        r_lr, r_film, n = evaluate(*F[reg])
        for t_, k in enumerate(K_ALL):
            rows.append([os.path.basename(path), t, *tinfo, reg, 'LR', k, float(Z_LEVELS[k]), r_lr[t_], n])
            rows.append([os.path.basename(path), t, *tinfo, reg, 'FiLM', k, float(Z_LEVELS[k]), r_film[t_], n])
    with open(out_csv + '.tmp', 'w', newline='') as f:
        w = csv.writer(f); w.writerow(_FIELDS); w.writerows(rows)
    os.replace(out_csv + '.tmp', out_csv)
    if F is not TRAIN_FIELDS:
        del F
    gc.collect()
    print(f'{tag}: {hours/24:+7.2f} days, done in {time.time() - t0:.0f} s', flush=True)

# ------------------------------------------------------------
# Summary: time of every snapshot, tables and figure
# ------------------------------------------------------------
_all = []
for (path, t) in TIME_SOURCES:
    tag = f'{os.path.splitext(os.path.basename(path))[0]}_t{t:02d}'
    f = os.path.join(TEMPORAL_DIR, f'r2_{tag}.csv')
    if os.path.exists(f):
        _all += list(csv.DictReader(open(f)))
with open(os.path.join(TEMPORAL_DIR, 'temporal_r2_all.csv'), 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=_FIELDS); w.writeheader(); w.writerows(_all)

_snaps = sorted({(r['file'], int(r['t_idx']), float(r['hours'])) for r in _all}, key=lambda s: s[2])
_tinfo = {(r['file'], int(r['t_idx'])): (float(r['days_since_init']), r['date']) for r in _all}
with open(os.path.join(TEMPORAL_DIR, 'temporal_snapshots.csv'), 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['file', 't_idx', 'days_since_init', 'date', 'days_from_training'])
    for (fn, t, h) in _snaps:
        w.writerow([fn, t, f'{_tinfo[(fn, t)][0]:.4f}', _tinfo[(fn, t)][1], f'{h/24:.4f}'])

_R = {(r['file'], int(r['t_idx']), r['region'], r['model'], int(r['k'])): float(r['r2']) for r in _all}
for reg in REGIONS:
    print(f'\nR² on {reg} (rows: snapshots, by days since the training snapshot)')
    hdr = (f'{"file":>18} {"init+d":>8} {"date":>16} {"days":>8} '
           + ' '.join(f'{"LR k="+str(k):>9} {"FiLM k="+str(k):>10}' for k in K_PRED_cnn))
    print(hdr); print('-' * len(hdr))
    for (fn, t, h) in _snaps:
        print(f'{fn:>18} {_tinfo[(fn, t)][0]:>8.2f} {_tinfo[(fn, t)][1]:>16} {h/24:>8.2f} '
              + ' '.join(f'{_R[(fn, t, reg, "LR", k)]:>9.3f} {_R[(fn, t, reg, "FiLM", k)]:>10.3f}'
                         for k in K_PRED_cnn))

C_LR, C_FILM = '#C2410C', '#1D4ED8'
fig, axs = plt.subplots(len(REGIONS), len(K_PRED_cnn), figsize=(4.2 * len(K_PRED_cnn), 3.6 * len(REGIONS)),
                        sharex=True, squeeze=False, constrained_layout=True)
_d = [s[2] / 24 for s in _snaps]
for a, reg in enumerate(REGIONS):
    for b, k in enumerate(K_PRED_cnn):
        ax = axs[a, b]
        for m, c, mk in (('LR', C_LR, 's'), ('FiLM', C_FILM, 'o')):
            ax.plot(_d, [_R[(fn, t, reg, m, k)] for (fn, t, h) in _snaps], color=c, marker=mk, ms=4, lw=1.5, label=m)
        ax.axvline(0, color='0.6', lw=1, ls=':')
        ax.set_title(f'{reg}, k={k}, z={Z_LEVELS[k]:.1f} m'); ax.grid(alpha=0.3)
        if a == len(REGIONS) - 1: ax.set_xlabel('days since training snapshot')
        if b == 0: ax.set_ylabel(r'$R^2$')
axs[0, 0].legend(frameon=False)
plt.savefig(os.path.join(TEMPORAL_DIR, 'temporal_r2.png'), dpi=200, bbox_inches='tight')
plt.savefig(os.path.join(TEMPORAL_DIR, 'temporal_r2.pdf'), bbox_inches='tight')
plt.show()
print(f'\nSaved in {TEMPORAL_DIR}: r2_<file>_tXX.csv, temporal_r2_all.csv, '
      f'temporal_snapshots.csv (time of every snapshot), temporal_r2.png/pdf')

## 6. Boundary layer, stratification and vertical correlation of w at every snapshot
For every snapshot of Section 5 (training snapshot + one raw file every `FILE_EVERY`, time index 0), reads from the file only the region boxes (Alboran, Transition and the SW Med training box) and computes, over the valid points of each box:
- the 10th, 50th and 90th percentiles of the KPP boundary-layer depth `hbl` (as in the paper, Sec. III B);
- the median temperature difference between the stencil level (k=2, -11.2 m) and the deepest target level (k=17, -116.6 m), as a simple measure of the upper-ocean stratification;
- the correlation between w at the stencil level (k=2) and w at every fixed level down to 400 m (as Fig. 6 of the paper);
- the standard deviation of w at every fixed level down to 400 m, to see how fast the amplitude of w changes with depth.

Stand-alone cell: it does not need the other cells (run it after a kernel restart, or after Section 1). It disables HDF5 file locking, which makes the files on /data hang on open. Calendar dates use 2010-01-01 00:00 as model start (training snapshot = 398.8125 days). Output: `temporal_validation/temporal_hbl.csv`, `temporal_wcorr.csv`, `temporal_wstd.csv` and the figures `temporal_hbl`, `temporal_wcorr`, `temporal_wstd` (png, pdf). Resumable: snapshots already in the CSV files are skipped. Every step is printed with the elapsed time.

In [ ]:
# ============================================================
# BOUNDARY LAYER, STRATIFICATION AND VERTICAL CORRELATION OF w
# AT EVERY SNAPSHOT (one pass over the files)
# Stand-alone cell: needs NO other cell of the notebook. Run it in a fresh
# kernel (Kernel -> Restart) or after Section 1. Same snapshots as the
# temporal validation (training snapshot + one raw file every FILE_EVERY,
# time index 0) and same region boxes. Reads from the files ONLY the boxes.
# For every snapshot and region, over the valid points of the box:
#   - hbl : 10th, 50th, 90th percentile of the KPP boundary-layer depth
#   - dT  : median of T(-11.2 m) - T(-116.6 m) (k=2 and k=17), a simple
#           measure of the upper-ocean stratification
#   - r(k): Pearson correlation between w at the stencil level (k=2,
#           -11.2 m) and w at every fixed level down to 400 m (as Fig. 6)
#   - std(k): standard deviation of w at every fixed level down to 400 m,
#           to see how fast the amplitude of w decays with depth
# Output (resumable, one row per snapshot and region):
#   temporal_validation/temporal_hbl.csv,   temporal_hbl.png/pdf
#   temporal_validation/temporal_wcorr.csv, temporal_wcorr.png/pdf
#   temporal_validation/temporal_wstd.csv,  temporal_wstd.png/pdf
# ============================================================
import os
# HDF5 file locking makes netCDF files on /data hang on open: disable it
# BEFORE netCDF4 is used (same as Section 1).
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
import glob, csv, time
import numpy as np
import netCDF4
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ===== PARAMETERS (same as the temporal-validation cell) =====
AVG_FILE     = './croco_avg.0656.nc'             # training day
GRID_FILE    = './grd_alb500sm.nc'
RAW_DIR      = '/data/geo/lamarca/ROMS500/raw'
FILE_EVERY   = 7
TEMPORAL_DIR = './temporal_validation'
REGIONS      = {'Alboran':    (35.70, 36.40, -3.60, -2.20),   # (lat_min, lat_max, lon_min, lon_max)
                'Transition': (36.40, 37.40, -1.50, -0.20),
                'Train':      (36.90, 38.30,  0.40,  2.00)}   # SW Med training box
MODEL_START_DATE = '2010-01-01 00:00'            # training snapshot (3 Feb 2011 19:30) = 398.8125 days
K_REF, K_BOT = 2, 17                             # stencil level and deepest target level
Z_MAX, MIN_PIX = 400.0, 100                      # correlation down to Z_MAX m, min valid points
# =============================================================

HBL_CSV = os.path.join(TEMPORAL_DIR, 'temporal_hbl.csv')
WC_CSV  = os.path.join(TEMPORAL_DIR, 'temporal_wcorr.csv')
WS_CSV  = os.path.join(TEMPORAL_DIR, 'temporal_wstd.csv')
os.makedirs(TEMPORAL_DIR, exist_ok=True)
_t_start = time.time()
def log(msg):
    print(f'[{time.time() - _t_start:7.1f} s] {msg}', flush=True)

log(f'working directory: {os.getcwd()}')
for p in (AVG_FILE, GRID_FILE):
    assert os.path.exists(p), f'file not found: {p}'

# --- grid (xcroco works on the inner rho points: NetCDF index = xcroco index + 1)
with netCDF4.Dataset(GRID_FILE) as g:
    H_nc   = np.asarray(g['h'][:], dtype=np.float64)
    lat_nc = np.asarray(g['lat_rho'][:], dtype=np.float64)
    lon_nc = np.asarray(g['lon_rho'][:], dtype=np.float64)
lat_in, lon_in, h_in = lat_nc[1:-1, 1:-1], lon_nc[1:-1, 1:-1], H_nc[1:-1, 1:-1]
log('grid read')

def z_sigma(zeta, h, sc, cs, hc):
    """CROCO Vtransform=2 depth of the sigma levels (same formula as gop.get_z)."""
    z0 = (hc * sc[:, None, None] + h[None] * cs[:, None, None]) / (hc + h[None])
    return zeta[None] + (zeta + h)[None] * z0

def interp_levels(z, f, z_new):
    """Linear interpolation of every column (z increasing along axis 0) onto z_new; NaN outside."""
    nz = z.shape[0]
    out = np.full((len(z_new),) + z.shape[1:], np.nan)
    for t, zt in enumerate(z_new):
        k = (z < zt).sum(0)
        ok = (k > 0) & (k < nz)
        kk = np.clip(k, 1, nz - 1)
        z0 = np.take_along_axis(z, (kk - 1)[None], 0)[0]; z1 = np.take_along_axis(z, kk[None], 0)[0]
        f0 = np.take_along_axis(f, (kk - 1)[None], 0)[0]; f1 = np.take_along_axis(f, kk[None], 0)[0]
        o = f0 + (zt - z0) / (z1 - z0) * (f1 - f0)
        o[~ok] = np.nan
        out[t] = o
    return out

def pearson(x, y):
    v = np.isfinite(x) & np.isfinite(y)
    if v.sum() < MIN_PIX:
        return np.nan
    a, b = x[v] - x[v].mean(), y[v] - y[v].mean()
    return float((a * b).sum() / np.sqrt((a * a).sum() * (b * b).sum()))

def read(d, name, t, sl3):
    return np.ma.filled(d[name][(t,) + sl3], np.nan).astype(np.float64)

# --- Z_LEVELS: sigma levels of the deepest column at the training snapshot (k=0 surface)
with netCDF4.Dataset(AVG_FILE) as d:
    hc = float(d.hc)
    sc_r, Cs_r = np.asarray(d.sc_r, float), np.asarray(d.Cs_r, float)
    jd, idp = np.unravel_index(np.nanargmax(h_in), h_in.shape)
    zeta_d = float(d['zeta'][0, jd + 1, idp + 1])
    tv = 'time' if 'time' in d.variables else 'ocean_time'
    T0_SEC = float(d[tv][0])
Z_LEVELS = np.sort(z_sigma(np.array([[zeta_d]]), np.array([[h_in[jd, idp]]]), sc_r, Cs_r, hc)[:, 0, 0])[::-1]
KALL = [k for k in range(len(Z_LEVELS)) if -Z_LEVELS[k] <= Z_MAX]
Z_TB = np.array([Z_LEVELS[K_REF], Z_LEVELS[K_BOT]])
log(f'Z_LEVELS: k={K_REF} at {Z_TB[0]:.2f} m, k={K_BOT} at {Z_TB[1]:.2f} m; '
    f'correlation on {len(KALL)} levels down to {-Z_LEVELS[KALL[-1]]:.1f} m')

# --- region boxes in NetCDF indices
BOX = {}
for reg, (la0, la1, lo0, lo1) in REGIONS.items():
    m = (lat_in >= la0) & (lat_in <= la1) & (lon_in >= lo0) & (lon_in <= lo1)
    jj, ii = np.where(m)
    j0, j1, i0, i1 = jj.min(), jj.max() + 1, ii.min(), ii.max() + 1
    BOX[reg] = dict(j0=j0 + 1, j1=j1 + 1, i0=i0 + 1, i1=i1 + 1, valid=m[j0:j1, i0:i1])
    log(f'{reg}: box {j1 - j0} x {i1 - i0} points, {int(m.sum())} valid')

# --- snapshots: training snapshot + one raw file every FILE_EVERY (time index 0)
_raw = sorted(glob.glob(os.path.join(RAW_DIR, 'croco_avg.*.nc')))
SOURCES = [(AVG_FILE, 0)] + [(f, 0) for f in _raw[::FILE_EVERY]
                             if os.path.basename(f) != os.path.basename(AVG_FILE)]
log(f'{len(_raw)} raw files in {RAW_DIR}; {len(SOURCES)} snapshots')
if _raw:
    with netCDF4.Dataset(_raw[0]) as d:
        miss = [v for v in ('zeta', 'w', 'temp', 'hbl') if v not in d.variables]
    log(f'variables missing in the raw files: {miss if miss else "none"}')

# --- csv files (resumable)
RCOLS = [f'r_k{k:02d}' for k in KALL]
KEYS = ['file', 't_idx', 'days_since_init', 'date', 'days_from_training', 'region']
F_HBL = KEYS + ['hbl_p10', 'hbl_p50', 'hbl_p90', 'dT_med', 'n_points']
F_WC  = KEYS + RCOLS
SCOLS = [f'std_k{k:02d}' for k in KALL]
F_WS  = KEYS + SCOLS
def done_set(path):
    if not os.path.exists(path):
        return set()
    return {(r['file'], r['t_idx'], r['region']) for r in csv.DictReader(open(path))}
done_h, done_w, done_s = done_set(HBL_CSV), done_set(WC_CSV), done_set(WS_CSV)
new_h, new_w, new_s = (not os.path.exists(p) for p in (HBL_CSV, WC_CSV, WS_CSV))
log(f'already done: {len(done_h)} rows in temporal_hbl.csv, {len(done_w)} in temporal_wcorr.csv, '
    f'{len(done_s)} in temporal_wstd.csv')

with open(HBL_CSV, 'a', newline='') as fh, open(WC_CSV, 'a', newline='') as fw, open(WS_CSV, 'a', newline='') as fs:
    wh, ww = csv.DictWriter(fh, fieldnames=F_HBL), csv.DictWriter(fw, fieldnames=F_WC)
    ws = csv.DictWriter(fs, fieldnames=F_WS)
    if new_h: wh.writeheader()
    if new_w: ww.writeheader()
    if new_s: ws.writeheader()
    for n_s, (path, t) in enumerate(SOURCES):
        name = os.path.basename(path)
        todo = [reg for reg in BOX if any((name, str(t), reg) not in dn for dn in (done_h, done_w, done_s))]
        if not todo:
            continue
        log(f'({n_s + 1}/{len(SOURCES)}) opening {name}')
        with netCDF4.Dataset(path) as d:
            tv = 'time' if 'time' in d.variables else 'ocean_time'
            sec = float(d[tv][t])
            date = (pd.Timestamp(MODEL_START_DATE) + pd.Timedelta(seconds=sec)).strftime('%Y-%m-%d %H:%M')
            hc = float(d.hc)
            sc_r, Cs_r = np.asarray(d.sc_r, float), np.asarray(d.Cs_r, float)
            sc_w, Cs_w = np.asarray(d.sc_w, float), np.asarray(d.Cs_w, float)
            for reg in todo:
                b = BOX[reg]
                sl = (slice(b['j0'], b['j1']), slice(b['i0'], b['i1']))
                sl3 = (slice(None),) + sl
                base = dict(file=name, t_idx=t, days_since_init=f'{sec / 86400:.4f}', date=date,
                            days_from_training=f'{(sec - T0_SEC) / 86400:.4f}', region=reg)
                zeta = read(d, 'zeta', t, sl)
                h = H_nc[sl]
                if (name, str(t), reg) not in done_h:
                    hbl = np.abs(read(d, 'hbl', t, sl))
                    ok = b['valid'] & np.isfinite(hbl) & (hbl > 0)
                    p10, p50, p90 = np.percentile(hbl[ok], [10, 50, 90])
                    Tz = interp_levels(z_sigma(zeta, h, sc_r, Cs_r, hc), read(d, 'temp', t, sl3), Z_TB)
                    dT = Tz[0] - Tz[1]
                    dT_med = float(np.median(dT[b['valid'] & np.isfinite(dT)]))
                    wh.writerow(dict(base, hbl_p10=f'{p10:.2f}', hbl_p50=f'{p50:.2f}', hbl_p90=f'{p90:.2f}',
                                     dT_med=f'{dT_med:.4f}', n_points=int(ok.sum()))); fh.flush()
                    log(f'   {date}  {reg:<11} hbl p10/p50/p90 = {p10:5.1f} {p50:5.1f} {p90:5.1f} m   '
                        f'dT = {dT_med:5.2f} C')
                if (name, str(t), reg) not in done_w or (name, str(t), reg) not in done_s:
                    W = interp_levels(z_sigma(zeta, h, sc_w, Cs_w, hc), read(d, 'w', t, sl3), Z_LEVELS[KALL])
                    W[:, ~b['valid']] = np.nan
                if (name, str(t), reg) not in done_s:
                    sd = [float(np.nanstd(W[n])) if np.isfinite(W[n]).sum() >= MIN_PIX else np.nan
                          for n in range(len(KALL))]
                    ws.writerow(dict(base, **{c: f'{v:.6e}' for c, v in zip(SCOLS, sd)})); fs.flush()
                    sk = dict(zip(KALL, sd))
                    log(f'   {date}  {reg:<11} std(w) k=7/k=2 = {sk[7] / sk[2]:5.2f}  k=12/k=2 = {sk[12] / sk[2]:5.2f}  '
                        f'k=17/k=2 = {sk[17] / sk[2]:5.2f}')
                if (name, str(t), reg) not in done_w:
                    x = W[KALL.index(K_REF)].ravel()
                    r = [pearson(x, W[n].ravel()) for n in range(len(KALL))]
                    ww.writerow(dict(base, **{c: f'{v:.4f}' for c, v in zip(RCOLS, r)})); fw.flush()
                    rk = dict(zip(KALL, r))
                    log(f'   {date}  {reg:<11} r(k=7) = {rk[7]:6.3f}  r(k=12) = {rk[12]:6.3f}  '
                        f'r(k=17) = {rk[17]:6.3f}')
log('all snapshots done')

# ---- figures
Hd = pd.read_csv(HBL_CSV, parse_dates=['date']).sort_values('date')
Wd = pd.read_csv(WC_CSV, parse_dates=['date']).sort_values('date')
Sd = pd.read_csv(WS_CSV, parse_dates=['date']).sort_values('date')
plt.rcParams.update({'font.size': 14, 'axes.titlesize': 14, 'axes.labelsize': 14,
                     'xtick.labelsize': 12, 'ytick.labelsize': 12, 'legend.fontsize': 12})
t_train = pd.Timestamp('2011-02-03 19:30')
title = lambda reg: {'Transition': 'Transition region', 'Train': 'SW Med (training)'}.get(reg, reg)
def date_axis(a):
    a.axvline(t_train, color='k', ls=':', lw=1.2)
    a.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
    a.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))

# (1) hbl percentiles and stratification against calendar date
fig, axs = plt.subplots(2, len(REGIONS), figsize=(9 * len(REGIONS), 8), sharex=True,
                        constrained_layout=True, squeeze=False)
for j, reg in enumerate(REGIONS):
    h = Hd[Hd.region == reg]
    ax = axs[0, j]
    ax.fill_between(h.date, h.hbl_p10, h.hbl_p90, color='0.8', label='10th-90th percentile')
    ax.plot(h.date, h.hbl_p50, 'k-o', ms=3, label='median')
    for k in (2, 7, 12, 17):                    # target levels
        ax.axhline(-Z_LEVELS[k], color='tab:blue', lw=0.8, ls='--')
    ax.set_ylim(max(Hd.hbl_p90.max(), 120) * 1.05, 0)
    ax.set_ylabel('Boundary-layer depth (m)')
    ax.set_title(title(reg))
    ax2 = axs[1, j]
    ax2.plot(h.date, h.dT_med, 'k-o', ms=3)
    ax2.set_ylabel(r'$T(-11.2\,{\rm m}) - T(-116.6\,{\rm m})$ (°C)')
    for a in (ax, ax2):
        date_axis(a); a.grid(alpha=0.3)
axs[0, 0].legend(loc='lower left')
plt.savefig(os.path.join(TEMPORAL_DIR, 'temporal_hbl.png'), dpi=150)
plt.savefig(os.path.join(TEMPORAL_DIR, 'temporal_hbl.pdf'))
plt.show()

# (2) depth-time map of the correlation, with hbl median and 90th percentile
fig, axs = plt.subplots(1, len(REGIONS), figsize=(9 * len(REGIONS), 5), sharey=True,
                        constrained_layout=True, squeeze=False)
depth = -Z_LEVELS[KALL]
for j, reg in enumerate(REGIONS):
    s, h = Wd[Wd.region == reg], Hd[Hd.region == reg]
    ax = axs[0, j]
    pc = ax.pcolormesh(s.date, depth, s[RCOLS].values.T, cmap='RdBu_r', vmin=-1, vmax=1, shading='nearest')
    ax.plot(h.date, h.hbl_p50, 'k-', lw=1.5, label='hbl median')
    ax.plot(h.date, h.hbl_p90, 'k--', lw=1.5, label='hbl 90th percentile')
    ax.set_ylim(Z_MAX, 0)
    ax.set_title(title(reg))
    date_axis(ax)
axs[0, 0].set_ylabel('Depth (m)')
axs[0, 0].legend(loc='lower left')
fig.colorbar(pc, ax=axs[0, -1], label=r'$r\,[w(-11.2\,{\rm m}),\,w(z)]$')
plt.savefig(os.path.join(TEMPORAL_DIR, 'temporal_wcorr.png'), dpi=150)
plt.savefig(os.path.join(TEMPORAL_DIR, 'temporal_wcorr.pdf'))
plt.show()

# (3) amplitude of w at depth relative to the stencil level: std(w_k) / std(w_2)
fig, axs = plt.subplots(1, len(REGIONS), figsize=(9 * len(REGIONS), 4.5), sharey=True,
                        constrained_layout=True, squeeze=False)
for j, reg in enumerate(REGIONS):
    s = Sd[Sd.region == reg]
    ax = axs[0, j]
    for k, c in ((7, 'tab:green'), (12, 'tab:orange'), (17, 'tab:purple')):
        ax.plot(s.date, s[f'std_k{k:02d}'] / s[f'std_k{K_REF:02d}'], '-o', ms=3, color=c,
                label=rf'$k={k}$ ($z={Z_LEVELS[k]:.1f}$ m)')
    ax.set_title(title(reg))
    date_axis(ax); ax.grid(alpha=0.3)
axs[0, 0].set_ylabel(r'$\sigma_w(z)\,/\,\sigma_w(-11.2\,{\rm m})$')
axs[0, 0].legend(loc='upper left')
plt.savefig(os.path.join(TEMPORAL_DIR, 'temporal_wstd.png'), dpi=150)
plt.savefig(os.path.join(TEMPORAL_DIR, 'temporal_wstd.pdf'))
plt.show()
log(f'saved {HBL_CSV}, {WC_CSV}, {WS_CSV} and the figures temporal_hbl / temporal_wcorr / temporal_wstd (png, pdf)')


## 7. Summer training and amplitude diagnostics
The same models as Section 5 (LR per level and FiLM, same box, same seed, same procedure) are trained again at a summer snapshot, `SUMMER_FILE` (default: 7 August 2011, 19:30 UTC, 185 days after the winter training snapshot, inside the summer collapse window). Both the winter and the summer models are then evaluated at all the snapshots of Section 5 on Alboran and Transition, to see whether the validation is symmetric: does the summer model fail in winter as the winter model fails in summer?

For every snapshot, region, model and level it also saves the amplitude diagnostics: correlation between predicted and true w, `std(predicted)/std(true)` and the normalized bias, with R² = 2 r std_ratio − std_ratio² − bias². The winter R² are checked against Section 5.

Needs Sections 1–5. The first cell below is `run_seed()`, copied unchanged from `notebook_nuredduna_seeds.ipynb` (training of one FiLM seed, about 0.4 h on the server). The output folders are named after the summer file, e.g. `cnn_summer_film_2136/` and `temporal_validation_summer_2136/` for `croco_avg.2136.nc`: changing `SUMMER_FILE` writes to new folders and never overwrites another day. For a given day the FiLM is trained only once (skipped if its `model_final.pt` exists), and the evaluation is resumable.

In [ ]:
# ============================================================
# ONE SEED: training + evaluation of the FiLM CNN, exactly as in the FiLM
# cell of notebook_nuredduna.ipynb (sections 4-7c + the depth-permutation
# test), but only NUMBERS are saved (no maps / figures / results.npz).
# The seed fixes everything random in a run: weight initialization, order
# of the patches in the batches, dropout masks.
# ============================================================
def _json_default(o):
    if hasattr(o, 'item'):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


def run_seed(seed, RUN_DIR):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    _t_seed0 = time.time()
    with open(os.path.join(RUN_DIR, 'config.json'), 'w') as f:
        json.dump(dict(_config, seed=seed), f, indent=2)
    print(f'Seed: {seed} | Run dir: {RUN_DIR} | Device: {device}')

    # ------------------------------------------------------------
    # 4. Model (FiLM) + patch-grouped training (conv1 once per patch,
    #    FiLM + conv2 + FC once per (patch, level) row)
    # ------------------------------------------------------------
    model_film = ZCondStencilCNN_FiLM(stencil_size_cnn).to(device)
    # Two L2 groups (Adam weight_decay = coupled L2 on the loss):
    #  - conv1 / conv2 / FC weights and biases : WD_OTHER
    #  - FiLM generator (to_gamma_beta)          : WD_FILM (much weaker)
    # L2 pulls every parameter towards 0 -- for the FiLM generator this means
    # w_gamma, w_beta, b_beta -> 0 and b_gamma -> 0 (NOT towards the identity
    # init gamma = 1, which is only the starting point, not a target).
    # Note: conv1 -> ReLU -> gamma*x + beta is invariant under
    # (conv1 / c, gamma * c) for any c > 0, so the WD_OTHER / WD_FILM ratio also
    # decides how the L2 penalty is split between conv1 and gamma: with
    # WD_OTHER >> WD_FILM the optimizer shrinks conv1 and grows gamma (trained
    # b_gamma ~ 2-4), i.e. conv1 is effectively regularized less than WD_OTHER
    # alone would suggest.
    film_params  = [p for n, p in model_film.named_parameters() if 'to_gamma_beta' in n]
    other_params = [p for n, p in model_film.named_parameters() if 'to_gamma_beta' not in n]
    optimizer = torch.optim.Adam([
        {'params': other_params, 'weight_decay': WD_OTHER},
        {'params': film_params,  'weight_decay': WD_FILM},  # value set in the PARAMETERS block
    ], lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10, min_lr=LR_MIN_cnn
    )
    criterion = nn.MSELoss()
    n_train = len(y_flat_ib_s)
    y_flat_ib_s_t = torch.tensor(y_flat_ib_s, dtype=torch.float32)
    z_flat_ib_s_t = torch.tensor(z_flat_ib_s, dtype=torch.float32).unsqueeze(1)
    # Patch-grouped batches: every batch takes whole patches (all their valid
    # levels together), so conv1 runs once per patch instead of once per row.
    # build_flat_training_rows returns rows sorted by patch (np.where on a 2D
    # mask is row-major), so each patch's rows are one contiguous block.
    assert np.all(np.diff(patch_idx_ib) >= 0), 'rows are expected to be grouped by patch'
    rows_per_patch = np.bincount(patch_idx_ib, minlength=len(patches_ib))
    row_start_of_patch = np.concatenate([[0], np.cumsum(rows_per_patch)[:-1]])
    patches_with_rows = np.nonzero(rows_per_patch)[0]
    PATCH_BATCH_cnn = max(1, ROW_BATCH_cnn // N_TARGET_cnn)   # ~ROW_BATCH_cnn rows per batch
    n_batches_per_epoch = (len(patches_with_rows) + PATCH_BATCH_cnn - 1) // PATCH_BATCH_cnn
    PROGRESS_EVERY_BATCHES = max(1, n_batches_per_epoch // 5)
    print(f'Training rows: {n_train} | patches: {len(patches_with_rows)} | batches per epoch: {n_batches_per_epoch} '
          f'(patches/batch={PATCH_BATCH_cnn}, ~{PATCH_BATCH_cnn * N_TARGET_cnn} rows/batch)')

    def _is_validation_epoch(ep):
        """True on the epochs where validation runs: dense at the start
        (VALIDATE_SCHEDULE), then every VALIDATE_EVERY_EPOCHS epochs."""
        for _until, _every in VALIDATE_SCHEDULE:
            if ep <= _until:
                return ep % _every == 0
        return ep % VALIDATE_EVERY_EPOCHS == 0

    history_film = {'epoch': [], 'train_loss': [], 'lr': []}
    for k_abs in K_PRED_cnn:
        history_film[f'val_r2_k{k_abs}'] = []       # Alboran (reference)
        history_film[f'val_r2_ag_k{k_abs}'] = []    # Algerian (2nd zone, tracked the same way)

    # Best-on-Alboran checkpoint: saved ONLY as a reference (model_best.pt and the
    # best-vs-final table) and used by the R² stopping criterion (b). All
    # evaluation/plots below use the FINAL model.
    _best_val_r2_mean = -np.inf
    _best_epoch = None
    _stop_reason = None

    for epoch in itertools.count():   # no fixed length: see the stopping criteria in PARAMETERS
        model_film.train()
        patch_perm = np.random.permutation(patches_with_rows)
        epoch_loss = 0.0
        t_epoch_start = time.time()

        for b, start in enumerate(range(0, len(patch_perm), PATCH_BATCH_cnn)):
            P_b = patch_perm[start:start + PATCH_BATCH_cnn]           # patches in this batch
            counts_b = rows_per_patch[P_b]
            offsets_b = np.concatenate([[0], np.cumsum(counts_b)[:-1]])
            idx_b = np.repeat(row_start_of_patch[P_b] - offsets_b, counts_b) + np.arange(counts_b.sum())
            inv_b = torch.as_tensor(np.repeat(np.arange(len(P_b)), counts_b), dtype=torch.long, device=device)
            xb = patches_ib_tensor[P_b].to(device)                    # one stencil per patch
            zb = z_flat_ib_s_t[idx_b].to(device)
            yb = y_flat_ib_s_t[idx_b].to(device)

            optimizer.zero_grad()
            pred = model_film.forward_grouped(xb, inv_b, zb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(idx_b)

            if epoch == 0 and (b + 1) % PROGRESS_EVERY_BATCHES == 0:
                elapsed = time.time() - t_epoch_start
                ram_gb = psutil.Process().memory_info().rss / 1e9
                print(f'  [epoch 1] batch {b+1}/{n_batches_per_epoch} | '
                      f'{elapsed:.1f}s elapsed | RAM: {ram_gb:.2f} GB')

        epoch_loss /= n_train
        epoch_time = time.time() - t_epoch_start
        scheduler.step(epoch_loss)
        current_lr = optimizer.param_groups[0]['lr']
        _lr_converged = current_lr <= LR_MIN_cnn * (1 + 1e-9)   # stopping criterion (a)
        ram_gb = psutil.Process().memory_info().rss / 1e9
        print(f'Epoch {epoch+1:4d} | Loss_train: {epoch_loss:.6e} | '
              f'lr: {current_lr:.2e} | {epoch_time:.1f}s/epoch | RAM: {ram_gb:.2f} GB')

        if _is_validation_epoch(epoch + 1) or _lr_converged:   # the last epoch is always validated
            model_film.eval()
            val_r2_this_epoch = {}
            for k_abs in K_PRED_cnn:
                t = k_abs - K_START_cnn
                col_true = y_all_na[:, t]
                valid_rows = ~np.isnan(col_true)
                if valid_rows.sum() == 0:
                    val_r2_this_epoch[k_abs] = np.nan
                    continue
                z_val = np.full(valid_rows.sum(), Z_LEVELS[k_abs], dtype=np.float32)
                z_val_s = (z_val - mean_z_cnn) / std_z_cnn
                patch_idx_val = np.where(valid_rows)[0]
                y_pred_s = predict_zcond_in_batches(model_film, patches_na_tensor, patch_idx_val, z_val_s, device)
                y_pred = y_pred_s * std_w_per_level[t] + mean_w_per_level[t]
                y_true = col_true[valid_rows]
                val_r2_this_epoch[k_abs] = r2_score(y_true, y_pred)
            # Algerian is tracked the same way, purely for visibility (the reference
            # best checkpoint below is keyed on Alboran only).
            val_r2_ag_this_epoch = {}
            for k_abs in K_PRED_cnn:
                t = k_abs - K_START_cnn
                col_true_ag = y_all_ag[:, t]
                valid_rows_ag = ~np.isnan(col_true_ag)
                if valid_rows_ag.sum() == 0:
                    val_r2_ag_this_epoch[k_abs] = np.nan
                    continue
                z_val_ag = np.full(valid_rows_ag.sum(), Z_LEVELS[k_abs], dtype=np.float32)
                z_val_ag_s = (z_val_ag - mean_z_cnn) / std_z_cnn
                patch_idx_val_ag = np.where(valid_rows_ag)[0]
                y_pred_ag_s = predict_zcond_in_batches(model_film, patches_ag_tensor, patch_idx_val_ag, z_val_ag_s, device)
                y_pred_ag = y_pred_ag_s * std_w_per_level[t] + mean_w_per_level[t]
                y_true_ag = col_true_ag[valid_rows_ag]
                val_r2_ag_this_epoch[k_abs] = r2_score(y_true_ag, y_pred_ag)
            history_film['epoch'].append(epoch + 1)
            history_film['train_loss'].append(epoch_loss)
            history_film['lr'].append(current_lr)
            for k_abs in K_PRED_cnn:
                history_film[f'val_r2_k{k_abs}'].append(val_r2_this_epoch[k_abs])
                history_film[f'val_r2_ag_k{k_abs}'].append(val_r2_ag_this_epoch[k_abs])
            r2_str = ' | '.join([f'R2(k={k})={val_r2_this_epoch[k]:.4f}' for k in K_PRED_cnn])
            r2_ag_str = ' | '.join([f'R2_AG(k={k})={val_r2_ag_this_epoch[k]:.4f}' for k in K_PRED_cnn])
            print(f'  >> Val check @ epoch {epoch+1}: {r2_str}')
            print(f'  >> Val check @ epoch {epoch+1} (Algerian): {r2_ag_str}')
            # Best checkpoint so far by mean R² across K_PRED_cnn on Alboran (ignoring
            # NaN): reference only (NOT used for evaluation) + stopping criterion (b).
            _vals = [v for v in val_r2_this_epoch.values() if not np.isnan(v)]
            if _vals:
                _mean_r2 = float(np.mean(_vals))
                if _mean_r2 > _best_val_r2_mean:
                    _best_val_r2_mean = _mean_r2
                    _best_epoch = epoch + 1
                    torch.save(model_film.state_dict(), os.path.join(RUN_DIR, 'model_best.pt'))
                elif (epoch + 1) - _best_epoch >= R2_PATIENCE_EPOCHS_cnn:
                    _stop_reason = (f'no improvement of the mean val R² on Alboran since epoch '
                                    f'{_best_epoch} ({R2_PATIENCE_EPOCHS_cnn} epochs)')

        if _lr_converged:
            _stop_reason = f'lr reached LR_MIN_cnn = {LR_MIN_cnn:.0e} (training converged)'
        if _stop_reason is not None:
            print(f'Stopping at epoch {epoch+1}: {_stop_reason}.')
            break

    # Always also save the final-epoch weights
    torch.save(model_film.state_dict(), os.path.join(RUN_DIR, 'model_final.pt'))
    print(f'Best-on-Alboran checkpoint (reference only): epoch {_best_epoch} '
          f'(mean val R² = {_best_val_r2_mean:.4f}) -> model_best.pt')
    print('Final-epoch weights (USED for all evaluation) -> model_final.pt')

    # ------------------------------------------------------------
    # R² of the FINAL model (model_film, used for everything below) and, for
    # reference, of model_best.pt -- side by side.
    # ------------------------------------------------------------
    def _r2_at_pred_levels(model):
        """R² at the K_PRED_cnn levels: Ibiza in-sample (train), Alboran, Algerian."""
        out = {}
        for t_k in K_PRED_cnn:
            t = t_k - K_START_cnn
            rows_t = np.where(level_idx_ib == t)[0]
            if len(rows_t) > 0:
                y_s = predict_zcond_in_batches(model, patches_ib_tensor, patch_idx_ib[rows_t],
                                               z_flat_ib_s[rows_t], device)
                out[('Ibiza (train)', t_k)] = r2_score(y_flat_ib[rows_t],
                                                       y_s * std_w_per_level[t] + mean_w_per_level[t])
            for zone, y_all_z, patches_z in (('Alboran', y_all_na, patches_na_tensor),
                                             ('Algerian', y_all_ag, patches_ag_tensor)):
                valid = ~np.isnan(y_all_z[:, t])
                if valid.sum() == 0:
                    out[(zone, t_k)] = np.nan
                    continue
                z_val = np.full(valid.sum(), Z_LEVELS[t_k], dtype=np.float32)
                z_val_s = (z_val - mean_z_cnn) / std_z_cnn
                y_s = predict_zcond_in_batches(model, patches_z, np.where(valid)[0], z_val_s, device)
                out[(zone, t_k)] = r2_score(y_all_z[valid, t],
                                            y_s * std_w_per_level[t] + mean_w_per_level[t])
        return out

    _final_epoch = epoch + 1
    _r2_final = _r2_at_pred_levels(model_film)

    # model_best.pt is evaluated in a SEPARATE model instance, so model_film keeps
    # the final weights for all evaluation/plots below.
    _r2_best = _r2_final
    if _best_epoch is not None:
        _model_best = ZCondStencilCNN_FiLM(stencil_size_cnn).to(device)
        _model_best.load_state_dict(torch.load(os.path.join(RUN_DIR, 'model_best.pt'), map_location=device))
        _r2_best = _r2_at_pred_levels(_model_best)
        del _model_best
    print(f'Using the FINAL model (epoch {_final_epoch}) for all evaluation/plots below.')
    print(f'\n=== model_best.pt (epoch {_best_epoch}, reference) vs model_final.pt (epoch {_final_epoch}, USED) ===')
    print(f'{"Zone":<15} {"Level":<7} {"z (m)":>8} {"R2 best":>9} {"R2 final":>9} {"final-best":>11}')
    print('-' * 64)
    _bf_rows = []
    for _zone in ('Ibiza (train)', 'Alboran', 'Algerian'):
        for t_k in K_PRED_cnn:
            _rb = _r2_best.get((_zone, t_k), np.nan)
            _rf = _r2_final.get((_zone, t_k), np.nan)
            print(f'{_zone:<15} k={t_k:<5} {Z_LEVELS[t_k]:>8.1f} {_rb:>9.4f} {_rf:>9.4f} {_rf - _rb:>+11.4f}')
            _bf_rows.append([_zone, t_k, float(Z_LEVELS[t_k]), _rb, _rf, _rf - _rb])
        _mb = np.nanmean([_r2_best.get((_zone, k), np.nan) for k in K_PRED_cnn])
        _mf = np.nanmean([_r2_final.get((_zone, k), np.nan) for k in K_PRED_cnn])
        print(f'{_zone:<15} {"mean":<7} {"":>8} {_mb:>9.4f} {_mf:>9.4f} {_mf - _mb:>+11.4f}')
        print('-' * 64)
    with open(os.path.join(RUN_DIR, 'best_vs_final_r2.csv'), 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['zone', 'level', 'z_m', 'r2_best', 'r2_final', 'final_minus_best'])
        writer.writerows(_bf_rows)
    print('Saved: best_vs_final_r2.csv')

    # Persist training/validation history to CSV
    with open(os.path.join(RUN_DIR, 'history.csv'), 'w', newline='') as f:
        writer = csv.writer(f)
        header = (['epoch', 'train_loss', 'lr'] + [f'val_r2_k{k}' for k in K_PRED_cnn]
                  + [f'val_r2_ag_k{k}' for k in K_PRED_cnn])
        writer.writerow(header)
        for i in range(len(history_film['epoch'])):
            row = [history_film['epoch'][i], history_film['train_loss'][i], history_film['lr'][i]]
            row += [history_film[f'val_r2_k{k}'][i] for k in K_PRED_cnn]
            row += [history_film[f'val_r2_ag_k{k}'][i] for k in K_PRED_cnn]
            writer.writerow(row)


    # ------------------------------------------------------------
    # 5. In-sample R² on Ibiza, per level
    # ------------------------------------------------------------
    y_pred_ib_s = predict_zcond_in_batches(model_film, patches_ib_tensor, patch_idx_ib, z_flat_ib_s, device)
    y_pred_ib = y_pred_ib_s * std_w_per_level[level_idx_ib] + mean_w_per_level[level_idx_ib]
    print(f'\n{"Level":<10} {"z (m)":>9} {"R²_CNN":>10} {"RMSE":>11} {"n_rows":>9}')
    print('-' * 54)
    ib_table_rows = []
    for t in range(N_TARGET_cnn):
        rows_t = level_idx_ib == t
        if rows_t.sum() == 0:
            continue
        r2_t = r2_score(y_flat_ib[rows_t], y_pred_ib[rows_t])
        rmse_t = np.sqrt(np.mean((y_flat_ib[rows_t] - y_pred_ib[rows_t])**2))
        k_level = K_START_cnn + t
        print(f'k={k_level:<7} {Z_LEVELS[k_level]:>9.1f} '
              f'{r2_t:>10.5f} {rmse_t:>11.3e} {rows_t.sum():>9d}')
        ib_table_rows.append([k_level, float(Z_LEVELS[k_level]), float(r2_t), float(rmse_t), int(rows_t.sum())])
    print('-' * 54)
    with open(os.path.join(RUN_DIR, 'ibiza_per_level_r2.csv'), 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['level', 'z_m', 'r2', 'rmse', 'n_rows'])
        writer.writerows(ib_table_rows)

    # ------------------------------------------------------------
    # 6. Alboran validation on K_PRED_cnn levels
    # ------------------------------------------------------------
    print(f'\n{"Level":<10} {"z (m)":>9} {"R²_CNN":>10} {"RMSE":>11}')
    print('-' * 44)
    w_cnn_maps = {}
    na_table_rows = []
    for k_abs in K_PRED_cnn:
        t = k_abs - K_START_cnn
        col_true = y_all_na[:, t]
        valid_rows = ~np.isnan(col_true)
        if valid_rows.sum() == 0:
            print(f'k={k_abs:<7} {Z_LEVELS[k_abs]:>9.1f} {"N/A":>10} {"N/A":>11}')
            na_table_rows.append([k_abs, float(Z_LEVELS[k_abs]), None, None])
            continue
        z_val = np.full(valid_rows.sum(), Z_LEVELS[k_abs], dtype=np.float32)
        z_val_s = (z_val - mean_z_cnn) / std_z_cnn
        patch_idx_val = np.where(valid_rows)[0]
        y_pred_s = predict_zcond_in_batches(model_film, patches_na_tensor, patch_idx_val, z_val_s, device)
        y_pred = y_pred_s * std_w_per_level[t] + mean_w_per_level[t]
        y_true = col_true[valid_rows]
        r2_t = r2_score(y_true, y_pred)
        rmse_t = np.sqrt(np.mean((y_true - y_pred)**2))
        print(f'k={k_abs:<7} {Z_LEVELS[k_abs]:>9.1f} {r2_t:>10.5f} {rmse_t:>11.3e}')
        na_table_rows.append([k_abs, float(Z_LEVELS[k_abs]), float(r2_t), float(rmse_t)])
        w_map = np.full((ny_na, nx_na), np.nan)
        w_map[centers_j_na[patch_idx_val], centers_i_na[patch_idx_val]] = y_pred
        w_cnn_maps[k_abs] = w_map
    print('-' * 44)
    with open(os.path.join(RUN_DIR, 'alboran_validation_r2.csv'), 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['level', 'z_m', 'r2', 'rmse'])
        writer.writerows(na_table_rows)

    # ------------------------------------------------------------
    # 6b. Algerian validation on K_PRED_cnn levels (2nd zone, same treatment)
    # ------------------------------------------------------------
    print(f'\n{"Level":<10} {"z (m)":>9} {"R²_CNN":>10} {"RMSE":>11}  (Algerian)')
    print('-' * 44)
    w_cnn_maps_ag = {}
    ag_table_rows = []
    for k_abs in K_PRED_cnn:
        t = k_abs - K_START_cnn
        col_true = y_all_ag[:, t]
        valid_rows = ~np.isnan(col_true)
        if valid_rows.sum() == 0:
            print(f'k={k_abs:<7} {Z_LEVELS[k_abs]:>9.1f} {"N/A":>10} {"N/A":>11}')
            ag_table_rows.append([k_abs, float(Z_LEVELS[k_abs]), None, None])
            continue
        z_val = np.full(valid_rows.sum(), Z_LEVELS[k_abs], dtype=np.float32)
        z_val_s = (z_val - mean_z_cnn) / std_z_cnn
        patch_idx_val = np.where(valid_rows)[0]
        y_pred_s = predict_zcond_in_batches(model_film, patches_ag_tensor, patch_idx_val, z_val_s, device)
        y_pred = y_pred_s * std_w_per_level[t] + mean_w_per_level[t]
        y_true = col_true[valid_rows]
        r2_t = r2_score(y_true, y_pred)
        rmse_t = np.sqrt(np.mean((y_true - y_pred)**2))
        print(f'k={k_abs:<7} {Z_LEVELS[k_abs]:>9.1f} {r2_t:>10.5f} {rmse_t:>11.3e}')
        ag_table_rows.append([k_abs, float(Z_LEVELS[k_abs]), float(r2_t), float(rmse_t)])
        w_map = np.full((ny_ag, nx_ag), np.nan)
        w_map[centers_j_ag[patch_idx_val], centers_i_ag[patch_idx_val]] = y_pred
        w_cnn_maps_ag[k_abs] = w_map
    print('-' * 44)
    with open(os.path.join(RUN_DIR, 'algerian_validation_r2.csv'), 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['level', 'z_m', 'r2', 'rmse'])
        writer.writerows(ag_table_rows)

    # ------------------------------------------------------------
    # 7. Interpolation check — predicting AT a depth strictly between two
    # trained levels (k=9, k=10), never seen exactly during training. There
    # is no real measurement at that in-between depth, so as a proxy for
    # "truth" we use the simplest possible estimate: the average of the true
    # w at the two bracketing levels. This is only a sanity check (w(z) is
    # not exactly linear between two levels), not a rigorous target — a
    # large disagreement would flag a discontinuity in what the network
    # learned between adjacent depths.
    # De-standardization at z_mid: mean_w/std_w linearly interpolated in z
    # between the two nearest training levels (w_stats_at_z).
    # ------------------------------------------------------------
    K_MID_LO, K_MID_HI = 9, 10
    t_mid_lo, t_mid_hi = K_MID_LO - K_START_cnn, K_MID_HI - K_START_cnn
    z_mid = float(np.mean(Z_LEVELS[[K_MID_LO, K_MID_HI]]))
    mean_w_mid, std_w_mid = w_stats_at_z(z_mid)
    z_mid_s = (z_mid - mean_z_cnn) / std_z_cnn
    print(f'\nInterpolation check: prediction at z={z_mid:.2f} m (midway between k={K_MID_LO} and k={K_MID_HI}, '
          f'never used as an exact training level)')

    # --- Alboran ---
    w_lo_na, w_hi_na = y_all_na[:, t_mid_lo], y_all_na[:, t_mid_hi]
    valid_mid_na = ~np.isnan(w_lo_na) & ~np.isnan(w_hi_na)
    y_true_mid_na = (w_lo_na[valid_mid_na] + w_hi_na[valid_mid_na]) / 2
    patch_idx_mid_na = np.where(valid_mid_na)[0]
    z_mid_s_arr_na = np.full(len(patch_idx_mid_na), z_mid_s, dtype=np.float32)
    y_pred_mid_na_s = predict_zcond_in_batches(model_film, patches_na_tensor, patch_idx_mid_na, z_mid_s_arr_na, device)
    y_pred_mid_na = y_pred_mid_na_s * std_w_mid + mean_w_mid
    r2_mid_na = float(r2_score(y_true_mid_na, y_pred_mid_na))
    rmse_mid_na = float(np.sqrt(np.mean((y_true_mid_na - y_pred_mid_na)**2)))
    print(f'Alboran  — vs (w_k{K_MID_LO}+w_k{K_MID_HI})/2 estimate: '
          f'R²={r2_mid_na:.5f} | RMSE={rmse_mid_na:.3e} | n={len(y_true_mid_na)}')

    # --- Algerian ---
    w_lo_ag, w_hi_ag = y_all_ag[:, t_mid_lo], y_all_ag[:, t_mid_hi]
    valid_mid_ag = ~np.isnan(w_lo_ag) & ~np.isnan(w_hi_ag)
    y_true_mid_ag = (w_lo_ag[valid_mid_ag] + w_hi_ag[valid_mid_ag]) / 2
    patch_idx_mid_ag = np.where(valid_mid_ag)[0]
    z_mid_s_arr_ag = np.full(len(patch_idx_mid_ag), z_mid_s, dtype=np.float32)
    y_pred_mid_ag_s = predict_zcond_in_batches(model_film, patches_ag_tensor, patch_idx_mid_ag, z_mid_s_arr_ag, device)
    y_pred_mid_ag = y_pred_mid_ag_s * std_w_mid + mean_w_mid
    r2_mid_ag = float(r2_score(y_true_mid_ag, y_pred_mid_ag))
    rmse_mid_ag = float(np.sqrt(np.mean((y_true_mid_ag - y_pred_mid_ag)**2)))
    print(f'Algerian — vs (w_k{K_MID_LO}+w_k{K_MID_HI})/2 estimate: '
          f'R²={r2_mid_ag:.5f} | RMSE={rmse_mid_ag:.3e} | n={len(y_true_mid_ag)}')

    # ------------------------------------------------------------
    # 7b. PURE extrapolation — z outside the range covered by training
    # De-standardization at that z: mean_w (linear fit in z) and std_w (linear
    # fit of log std_w in z, then exp) on the N_EXTRAP_FIT_cnn deepest training
    # levels (w_stats_at_z).
    # ------------------------------------------------------------
    z_top_train = float(Z_LEVELS[K_START_cnn])                     # shallowest training level
    z_bot_train = float(Z_LEVELS[K_START_cnn + N_TARGET_cnn - 1])  # deepest training level
    EXTRAPOLATE_Z_cnn = float(Z_LEVELS[K_EXTRAPOLATE_cnn])
    print(f'\nPure extrapolation at k={K_EXTRAPOLATE_cnn} (z={EXTRAPOLATE_Z_cnn:.2f} m) '
          f'| training range: [{z_top_train:.1f}, {z_bot_train:.1f}] m — out of range, no guarantee')
    z_extra_s = (np.full(len(patches_na), EXTRAPOLATE_Z_cnn, dtype=np.float32) - mean_z_cnn) / std_z_cnn
    y_extra_s = predict_zcond_in_batches(model_film, patches_na_tensor, np.arange(len(patches_na)), z_extra_s, device)
    mean_w_extra, std_w_extra = w_stats_at_z(EXTRAPOLATE_Z_cnn)
    print(f'Target stats at z={EXTRAPOLATE_Z_cnn:.2f} m (fit on the {N_EXTRAP_FIT_cnn} deepest training levels: mean linear, log std linear): '
          f'mean_w={mean_w_extra:.3e}, std_w={std_w_extra:.3e} m/s '
          f'(deepest training level: mean_w={mean_w_per_level[-1]:.3e}, std_w={std_w_per_level[-1]:.3e})')
    y_extra = y_extra_s * std_w_extra + mean_w_extra
    print(f'w extrapolated on {len(y_extra)} Alboran points at k={K_EXTRAPOLATE_cnn} — '
          f'range: [{y_extra.min():.3e}, {y_extra.max():.3e}] m/s')
    r2_extra, rmse_extra, n_extra = None, None, 0
    if K_EXTRAPOLATE_cnn < W_NA.shape[0]:
        y_true_extra = W_NA[K_EXTRAPOLATE_cnn, centers_j_na, centers_i_na]
        valid_extra = ~np.isnan(y_true_extra)
        if valid_extra.sum() > 0:
            r2_extra = float(r2_score(y_true_extra[valid_extra], y_extra[valid_extra]))
            rmse_extra = float(np.sqrt(np.mean((y_true_extra[valid_extra] - y_extra[valid_extra])**2)))
            n_extra = int(valid_extra.sum())
            print(f'Comparison with the true value at k={K_EXTRAPOLATE_cnn} (never used in training): '
                  f'R²={r2_extra:.5f} | RMSE={rmse_extra:.3e} | n={n_extra}')
        else:
            print(f'No valid true data at k={K_EXTRAPOLATE_cnn} for comparison.')
    else:
        print(f'k={K_EXTRAPOLATE_cnn} is beyond the data available in W_NA — no comparison possible.')

    # ------------------------------------------------------------
    # 7c. PURE extrapolation on Algerian (same check, 2nd zone)
    # ------------------------------------------------------------
    z_extra_ag_s = (np.full(len(patches_ag), EXTRAPOLATE_Z_cnn, dtype=np.float32) - mean_z_cnn) / std_z_cnn
    y_extra_ag_s = predict_zcond_in_batches(model_film, patches_ag_tensor, np.arange(len(patches_ag)), z_extra_ag_s, device)
    y_extra_ag = y_extra_ag_s * std_w_extra + mean_w_extra
    print(f'w extrapolated on {len(y_extra_ag)} Algerian points at k={K_EXTRAPOLATE_cnn} — '
          f'range: [{y_extra_ag.min():.3e}, {y_extra_ag.max():.3e}] m/s')
    r2_extra_ag, rmse_extra_ag, n_extra_ag = None, None, 0
    if K_EXTRAPOLATE_cnn < W_AG.shape[0]:
        y_true_extra_ag = W_AG[K_EXTRAPOLATE_cnn, centers_j_ag, centers_i_ag]
        valid_extra_ag = ~np.isnan(y_true_extra_ag)
        if valid_extra_ag.sum() > 0:
            r2_extra_ag = float(r2_score(y_true_extra_ag[valid_extra_ag], y_extra_ag[valid_extra_ag]))
            rmse_extra_ag = float(np.sqrt(np.mean((y_true_extra_ag[valid_extra_ag] - y_extra_ag[valid_extra_ag])**2)))
            n_extra_ag = int(valid_extra_ag.sum())
            print(f'Comparison with the true value at k={K_EXTRAPOLATE_cnn} on Algerian (never used in training): '
                  f'R\u00b2={r2_extra_ag:.5f} | RMSE={rmse_extra_ag:.3e} | n={n_extra_ag}')
        else:
            print(f'No valid true data at k={K_EXTRAPOLATE_cnn} for comparison (Algerian).')
    else:
        print(f'k={K_EXTRAPOLATE_cnn} is beyond the data available in W_AG — no comparison possible.')


    # ------------------------------------------------------------
    # Test 1 — does the network actually use z_target? (depth permutation)
    # For each predicted level, compares validation R² with the true z vs a
    # randomly WRONG z (drawn from the other training levels). Similar R² ->
    # the network ignores depth; much lower R² with the wrong z -> depth is used.
    #
    # Run right after the FiLM cell (Section 10), in the same kernel: it uses
    # model_film (FINAL-epoch weights), the Alboran/Algerian
    # patches and the standardization constants defined there.
    #
    # Note on de-standardization: both predictions are converted back to m/s
    # with the stats of the TRUE level t. The per-level stats are not part of
    # the network, so this isolates what the network itself does with z; using
    # the wrong level's stats would make R² collapse even for a network that
    # ignores z completely.
    # ------------------------------------------------------------
    Z_TEST_ZONES = {
        'Alboran':  (y_all_na, patches_na_tensor),
        'Algerian': (y_all_ag, patches_ag_tensor),
    }
    Z_TEST_REPEATS = 5


    def r2_with_shuffled_z(y_all_zone, patches_zone_tensor, k_abs, n_repeats=Z_TEST_REPEATS, seed=0):
        t = k_abs - K_START_cnn
        col_true = y_all_zone[:, t]
        valid_rows = ~np.isnan(col_true)
        n_valid = int(valid_rows.sum())
        if n_valid == 0:
            return None
        y_true = col_true[valid_rows]
        patch_idx_val = np.where(valid_rows)[0]

        def r2_for(z_raw):
            z_s = ((z_raw - mean_z_cnn) / std_z_cnn).astype(np.float32)
            y_pred_s = predict_zcond_in_batches(model_film, patches_zone_tensor, patch_idx_val, z_s, device)
            y_pred = y_pred_s * std_w_per_level[t] + mean_w_per_level[t]
            return r2_score(y_true, y_pred)

        # true depth (baseline)
        r2_true = r2_for(np.full(n_valid, Z_LEVELS[k_abs], dtype=np.float64))

        # wrong depth: each row gets a random level among the OTHER training levels
        other_levels = np.delete(np.ascontiguousarray(z_levels_ib, dtype=np.float64), t)
        rng = np.random.default_rng(seed + k_abs)
        r2_shuffled = [r2_for(rng.choice(other_levels, size=n_valid)) for _ in range(n_repeats)]
        return r2_true, float(np.mean(r2_shuffled)), float(np.std(r2_shuffled)), n_valid


    z_test_rows = []
    for zone_name, (y_all_zone, patches_zone_tensor) in Z_TEST_ZONES.items():
        print(f'\n=== {zone_name} ===')
        print(f'{"Level":<8} {"z (m)":>8} {"R2_true_z":>10} {"R2_wrong_z (mean +- std)":>27} {"drop":>8} {"N":>7}')
        print('-' * 74)
        for k_abs in K_PRED_cnn:
            res = r2_with_shuffled_z(y_all_zone, patches_zone_tensor, k_abs)
            if res is None:
                print(f'k={k_abs:<6} no valid rows')
                continue
            r2_true, r2_sh_mean, r2_sh_std, n_valid = res
            drop = r2_true - r2_sh_mean
            print(f'k={k_abs:<6} {Z_LEVELS[k_abs]:>8.1f} {r2_true:>10.4f} {r2_sh_mean:>15.4f} +- {r2_sh_std:.4f} '
                  f'{drop:>8.4f} {n_valid:>7d}')
            z_test_rows.append([zone_name, k_abs, float(Z_LEVELS[k_abs]), r2_true, r2_sh_mean, r2_sh_std, drop, n_valid])
        print('-' * 74)

    print('\nSimilar R2 between the two columns (drop ~ 0) -> the network IGNORES depth.')
    print('Much lower R2_wrong_z (large drop) -> depth is actually used.')

    _z_test_path = os.path.join(RUN_DIR, 'z_permutation_test.csv')
    with open(_z_test_path, 'w', newline='') as _f:
        _w = csv.writer(_f)
        _w.writerow(['zone', 'level', 'z_m', 'r2_true_z', 'r2_wrong_z_mean', 'r2_wrong_z_std', 'drop', 'n_rows'])
        _w.writerows(z_test_rows)
    print(f'Saved: {_z_test_path}')

    # ---- everything numeric of this seed in one json (read by the ensemble statistics cell)
    _result = dict(
        seed=seed,
        final_epoch=_final_epoch, stop_reason=str(_stop_reason),
        best_epoch=_best_epoch, best_val_r2_mean=_best_val_r2_mean,
        final_train_loss=history_film['train_loss'][-1] if history_film['train_loss'] else None,
        r2_final={f'{zone}|{k}': v for (zone, k), v in _r2_final.items()},
        r2_best_reference={f'{zone}|{k}': v for (zone, k), v in _r2_best.items()},
        ibiza_per_level=[dict(level=r[0], z_m=r[1], r2=r[2], rmse=r[3], n=r[4]) for r in ib_table_rows],
        interpolation=dict(k_lo=K_MID_LO, k_hi=K_MID_HI, z_m=z_mid, alboran=r2_mid_na, algerian=r2_mid_ag),
        extrapolation=dict(k=K_EXTRAPOLATE_cnn, z_m=EXTRAPOLATE_Z_cnn, alboran=r2_extra, algerian=r2_extra_ag,
                           mean_w=mean_w_extra, std_w=std_w_extra),
        z_permutation=[dict(zone=r[0], level=r[1], z_m=r[2], r2_true_z=r[3], r2_wrong_z_mean=r[4],
                            r2_wrong_z_std=r[5], drop=r[6], n=r[7]) for r in z_test_rows],
        duration_h=(time.time() - _t_seed0) / 3600,
    )
    with open(os.path.join(RUN_DIR, 'result.json'), 'w') as f:
        json.dump(_result, f, indent=2, default=_json_default)
    print(f'Saved: {os.path.join(RUN_DIR, "result.json")}')
    return _result


In [ ]:
# ============================================================
# SUMMER TRAINING + AMPLITUDE DIAGNOSTICS
# Needs Sections 1-5 and the cell above (run_seed). Steps:
#  1. winter models (as Section 5): LR refitted on the training box at the
#     training snapshot, FiLM = model_final.pt of seed SEED_FILM;
#  2. summer models: same box, same procedure, same seed, but trained at a
#     summer snapshot (SUMMER_FILE). The FiLM is trained with run_seed(),
#     identical to notebook_nuredduna_seeds.ipynb (stopping decided on
#     Alboran at the same summer snapshot). Skipped if already trained.
#  3. both pairs of models are evaluated at every snapshot of Section 5, on
#     Alboran and Transition, at all 16 target levels. For every
#     (snapshot, region, training, model, level):
#       r2        : R²
#       r_pred    : correlation between predicted and true w
#       std_ratio : std(predicted w) / std(true w)
#       bias_norm : (mean predicted w - mean true w) / std(true w)
#     so that R² = 2 r_pred std_ratio - std_ratio² - bias_norm²
#     (R² < 0 can come from a low correlation or from a wrong amplitude).
#  4. check: the winter R² must reproduce Section 5 (r2_<file>.csv).
# Output (resumable, one CSV per snapshot): temporal_validation_summer_<NNNN>/
# (NNNN = number of the summer file; the FiLM in cnn_summer_film_<NNNN>/)
#   eval_<file>_tXX.csv, eval_all.csv, temporal_r2_winter_summer.png/pdf,
#   temporal_amplitude.png/pdf
# ============================================================
import os, glob, csv, json, time, gc
import netCDF4
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# ===== PARAMETERS =====
RAW_DIR_S     = '/data/geo/lamarca/ROMS500/raw'
FILE_EVERY_S  = 7
SUMMER_FILE   = os.path.join(RAW_DIR_S, 'croco_avg.2136.nc')   # 7 Aug 2011, 19:30 UTC at time index 0
SUMMER_T      = 0                                              # (185 days after the winter training snapshot)
SEED_S        = 261                                            # same seed as the winter FiLM (Section 5)
WINTER_FILM   = os.path.join('./cnn_seeds_film', f'seed_{SEED_S:03d}', 'model_final.pt')
# output folders named after the summer file (e.g. 2136): changing SUMMER_FILE
# writes to new folders and never overwrites the results of another day
_STAG         = os.path.splitext(os.path.basename(SUMMER_FILE))[0].split('.')[-1]
SUMMER_RUN    = os.path.join(f'./cnn_summer_film_{_STAG}', f'seed_{SEED_S:03d}')
OUT_DIR       = f'./temporal_validation_summer_{_STAG}'
SEC5_DIR      = './temporal_validation'                        # Section 5 output, for the check
REG_T         = ['Alboran', 'Transition']
START_DATE    = '2010-01-01 00:00'                             # model time origin
# ======================
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(SUMMER_RUN, exist_ok=True)
_t_s = time.time()
def logs(msg):
    print(f'[{time.time() - _t_s:7.1f} s] {msg}', flush=True)

# Z_LEVELS of Section 3 (Section 6 redefines Z_LEVELS with netCDF4: restore the original)
Z_LEVELS = Z_ASC[::-1].copy()
_KT = list(range(K_START_cnn, K_START_cnn + N_TARGET_cnn))
_z_tr_s = Z_LEVELS[K_START_cnn:K_START_cnn + N_TARGET_cnn]
PRED_BATCH_S = 16 * PRED_BATCH_cnn


def _lr_feat(p):
    n = len(p)
    return np.concatenate([p[:, 0].reshape(n, -1), p[:, 1].reshape(n, -1)], axis=1)


def set_training_globals(F):
    """Parts 1-3b of the FiLM setup cell (Section 4) with the fields F
    ('Train'/'Alboran'/'Transition' -> (U, V, W)): patches, targets and
    standardization, i.e. everything run_seed() reads."""
    global U_IB, V_IB, W_IB, U_NA, V_NA, W_NA, U_AG, V_AG, W_AG, z_levels_ib
    global patches_ib, y_all_ib, patch_idx_ib, level_idx_ib, z_flat_ib, y_flat_ib
    global mean_u_cnn, std_u_cnn, mean_v_cnn, std_v_cnn, mean_z_cnn, std_z_cnn
    global mean_w_per_level, std_w_per_level, patches_ib_s, z_flat_ib_s, y_flat_ib_s, patches_ib_tensor
    global _z_train, _asc
    global ny_na, nx_na, patches_na, y_all_na, centers_j_na, centers_i_na, patches_na_s, patches_na_tensor
    global ny_ag, nx_ag, patches_ag, y_all_ag, centers_j_ag, centers_i_ag, patches_ag_s, patches_ag_tensor
    U_IB, V_IB, W_IB = F['Train']
    U_NA, V_NA, W_NA = F['Alboran']
    U_AG, V_AG, W_AG = F['Transition']
    ks = slice(K_START_cnn, K_START_cnn + N_TARGET_cnn)
    z_levels_ib = Z_LEVELS[ks]
    patches_ib, y_all_ib, _, _ = extract_patches_with_all_levels(U_IB[K_START_cnn], V_IB[K_START_cnn], W_IB[ks], r_cnn)
    patch_idx_ib, level_idx_ib, z_flat_ib, y_flat_ib = build_flat_training_rows(y_all_ib, z_levels_ib)
    mean_u_cnn, std_u_cnn = patches_ib[:, 0].mean(), patches_ib[:, 0].std()
    mean_v_cnn, std_v_cnn = patches_ib[:, 1].mean(), patches_ib[:, 1].std()
    patches_ib_s = patches_ib.copy().astype(np.float32)
    patches_ib_s[:, 0] = (patches_ib[:, 0] - mean_u_cnn) / std_u_cnn
    patches_ib_s[:, 1] = (patches_ib[:, 1] - mean_v_cnn) / std_v_cnn
    mean_z_cnn, std_z_cnn = z_levels_ib.mean(), z_levels_ib.std()
    z_flat_ib_s = (z_flat_ib - mean_z_cnn) / std_z_cnn
    mean_w_per_level = np.array([np.nanmean(y_all_ib[:, t]) for t in range(N_TARGET_cnn)])
    std_w_per_level = np.array([np.nanstd(y_all_ib[:, t]) for t in range(N_TARGET_cnn)]) + 1e-10
    y_flat_ib_s = (y_flat_ib - mean_w_per_level[level_idx_ib]) / std_w_per_level[level_idx_ib]
    patches_ib_tensor = torch.tensor(patches_ib_s, dtype=torch.float32)
    _z_train = np.asarray(z_levels_ib, dtype=np.float64)
    _asc = np.argsort(_z_train)
    ny_na, nx_na = U_NA[K_START_cnn].shape
    patches_na, y_all_na, centers_j_na, centers_i_na = extract_patches_with_all_levels(
        U_NA[K_START_cnn], V_NA[K_START_cnn], W_NA[ks], r_cnn)
    patches_na_s = patches_na.copy().astype(np.float32)
    patches_na_s[:, 0] = (patches_na[:, 0] - mean_u_cnn) / std_u_cnn
    patches_na_s[:, 1] = (patches_na[:, 1] - mean_v_cnn) / std_v_cnn
    patches_na_tensor = torch.tensor(patches_na_s, dtype=torch.float32)
    ny_ag, nx_ag = U_AG[K_START_cnn].shape
    patches_ag, y_all_ag, centers_j_ag, centers_i_ag = extract_patches_with_all_levels(
        U_AG[K_START_cnn], V_AG[K_START_cnn], W_AG[ks], r_cnn)
    patches_ag_s = patches_ag.copy().astype(np.float32)
    patches_ag_s[:, 0] = (patches_ag[:, 0] - mean_u_cnn) / std_u_cnn
    patches_ag_s[:, 1] = (patches_ag[:, 1] - mean_v_cnn) / std_v_cnn
    patches_ag_tensor = torch.tensor(patches_ag_s, dtype=torch.float32)
    print(f'  training globals set: {len(patches_ib)} training patches, {len(y_flat_ib)} rows', flush=True)


def make_bundle(film_path):
    """LR per level refitted on the CURRENT training globals + FiLM weights, with
    the standardization constants of the same training snapshot."""
    X = _lr_feat(patches_ib)
    lr = {}
    for t in range(N_TARGET_cnn):
        ok = ~np.isnan(y_all_ib[:, t])
        lr[t] = LinearRegression().fit(X[ok], y_all_ib[ok, t])
    m = ZCondStencilCNN_FiLM(stencil_size_cnn).to(device)
    m.load_state_dict(torch.load(film_path, map_location=device))
    m.eval()
    return dict(lr=lr, film=m, mu_u=mean_u_cnn, sd_u=std_u_cnn, mu_v=mean_v_cnn, sd_v=std_v_cnn,
                mu_z=mean_z_cnn, sd_z=std_z_cnn, mu_w=mean_w_per_level.copy(), sd_w=std_w_per_level.copy())


def _metrics(y, yp):
    sy = y.std()
    return dict(r2=float(r2_score(y, yp)), r_pred=float(np.corrcoef(y, yp)[0, 1]),
                std_ratio=float(yp.std() / sy), bias_norm=float((yp.mean() - y.mean()) / sy),
                std_true=float(sy), n=int(len(y)))


def evaluate_bundle(B, U, V, W):
    """Same computation as evaluate() of Section 5, with the constants of bundle B,
    plus the amplitude diagnostics. Returns {(model, k): metrics}."""
    Ws = W[K_START_cnn:K_START_cnn + N_TARGET_cnn]
    p, y_all, _, _ = extract_patches_with_all_levels(U[K_START_cnn], V[K_START_cnn], Ws, r_cnn)
    X = _lr_feat(p)
    ps = p.astype(np.float32).copy()
    ps[:, 0] = (p[:, 0] - B['mu_u']) / B['sd_u']
    ps[:, 1] = (p[:, 1] - B['mu_v']) / B['sd_v']
    pidx, lidx, zf, yf = build_flat_training_rows(y_all, _z_tr_s)
    ys = predict_zcond_in_batches(B['film'], torch.tensor(ps), pidx, (zf - B['mu_z']) / B['sd_z'],
                                  device, batch_size=PRED_BATCH_S)
    yp = ys * B['sd_w'][lidx] + B['mu_w'][lidx]
    out = {}
    for t, k in enumerate(_KT):
        ok = ~np.isnan(y_all[:, t])
        out[('LR', k)] = _metrics(y_all[ok, t], B['lr'][t].predict(X[ok]))
        sel = lidx == t
        out[('FiLM', k)] = _metrics(yf[sel].astype(np.float64), yp[sel].astype(np.float64))
    return out


# ------------------------------------------------------------
# 1. Winter models
# ------------------------------------------------------------
logs('winter models (training snapshot of Section 5)')
set_training_globals(TRAIN_FIELDS)
BUNDLES = {'winter': make_bundle(WINTER_FILM)}

# ------------------------------------------------------------
# 2. Summer models
# ------------------------------------------------------------
with netCDF4.Dataset(SUMMER_FILE) as _d:
    _tv = 'time' if 'time' in _d.variables else 'ocean_time'
    _sec_s = float(_d[_tv][SUMMER_T])
SUMMER_DATE = pd.Timestamp(START_DATE) + pd.Timedelta(seconds=_sec_s)
logs(f'summer snapshot: {os.path.basename(SUMMER_FILE)}, t={SUMMER_T}, {SUMMER_DATE:%Y-%m-%d %H:%M}')
SUMMER_FIELDS = fields_at_np(SUMMER_FILE, SUMMER_T, ['Train', 'Alboran', 'Transition'])
set_training_globals(SUMMER_FIELDS)
_summer_pt = os.path.join(SUMMER_RUN, 'model_final.pt')
if os.path.exists(_summer_pt):
    logs(f'summer FiLM already trained: {_summer_pt}')
else:
    logs(f'training the summer FiLM (seed {SEED_S}) in {SUMMER_RUN}')
    _res_s = run_seed(SEED_S, SUMMER_RUN)
    logs(f'summer FiLM trained in {_res_s["duration_h"]:.2f} h, stop: {_res_s["stop_reason"]}')
BUNDLES['summer'] = make_bundle(_summer_pt)
with open(os.path.join(SUMMER_RUN, 'summer_snapshot.json'), 'w') as f:
    json.dump(dict(file=os.path.basename(SUMMER_FILE), t_idx=SUMMER_T,
                   date=f'{SUMMER_DATE:%Y-%m-%d %H:%M}'), f, indent=2)
set_training_globals(TRAIN_FIELDS)            # restore the winter globals of Section 4
logs('winter globals restored')

# ------------------------------------------------------------
# 3. Evaluation at every snapshot (resumable: one CSV per snapshot)
# ------------------------------------------------------------
_raw_s = sorted(glob.glob(os.path.join(RAW_DIR_S, 'croco_avg.*.nc')))
SOURCES_S = [(AVG_FILE, T_IDX)] + [(f, 0) for f in _raw_s[::FILE_EVERY_S]
                                   if os.path.basename(f) != os.path.basename(AVG_FILE)]
if (SUMMER_FILE, SUMMER_T) not in SOURCES_S:
    SOURCES_S.append((SUMMER_FILE, SUMMER_T))
logs(f'{len(SOURCES_S)} snapshots to evaluate')
with netCDF4.Dataset(AVG_FILE) as _d:
    _tv = 'time' if 'time' in _d.variables else 'ocean_time'
    _T0_S = float(_d[_tv][T_IDX])

FIELDS_E = ['file', 't_idx', 'days_since_init', 'date', 'days_from_training', 'region', 'training',
            'model', 'k', 'z_m', 'r2', 'r_pred', 'std_ratio', 'bias_norm', 'std_true', 'n']
_n_bad = 0
for n_s, (path, t) in enumerate(SOURCES_S):
    tag = f'{os.path.splitext(os.path.basename(path))[0]}_t{t:02d}'
    out_csv = os.path.join(OUT_DIR, f'eval_{tag}.csv')
    if os.path.exists(out_csv):
        continue
    t0 = time.time()
    with netCDF4.Dataset(path) as _d:
        _tv = 'time' if 'time' in _d.variables else 'ocean_time'
        sec = float(_d[_tv][t])
    date = (pd.Timestamp(START_DATE) + pd.Timedelta(seconds=sec)).strftime('%Y-%m-%d %H:%M')
    F = TRAIN_FIELDS if (path, t) == (AVG_FILE, T_IDX) else fields_at_np(path, t, REG_T)
    rows = []
    for reg in REG_T:
        for tr, B in BUNDLES.items():
            res = evaluate_bundle(B, *F[reg])
            for (model, k), m in res.items():
                rows.append(dict(file=os.path.basename(path), t_idx=t, days_since_init=f'{sec / 86400:.4f}',
                                 date=date, days_from_training=f'{(sec - _T0_S) / 86400:.4f}', region=reg,
                                 training=tr, model=model, k=k, z_m=f'{Z_LEVELS[k]:.2f}', **m))
    # check: winter R² = Section 5
    _ref = os.path.join(SEC5_DIR, f'r2_{tag}.csv')
    if os.path.exists(_ref):
        R5 = {(r['region'], r['model'], int(r['k'])): float(r['r2']) for r in csv.DictReader(open(_ref))}
        for r in rows:
            if r['training'] == 'winter':
                ref = R5.get((r['region'], r['model'], r['k']))
                if ref is not None and abs(ref - r['r2']) > 1e-3:
                    _n_bad += 1
                    print(f'   MISMATCH {tag} {r["region"]} {r["model"]} k={r["k"]}: {r["r2"]:.4f} vs Section 5 {ref:.4f}')
    with open(out_csv + '.tmp', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=FIELDS_E); w.writeheader(); w.writerows(rows)
    os.replace(out_csv + '.tmp', out_csv)
    if F is not TRAIN_FIELDS:
        del F
    gc.collect()
    g = {(r['region'], r['training'], r['model'], r['k']): r for r in rows}
    logs(f'({n_s + 1}/{len(SOURCES_S)}) {date}  Transition k=7 R² LR/FiLM: winter '
         f'{g[("Transition", "winter", "LR", 7)]["r2"]:6.2f} {g[("Transition", "winter", "FiLM", 7)]["r2"]:6.2f} | summer '
         f'{g[("Transition", "summer", "LR", 7)]["r2"]:6.2f} {g[("Transition", "summer", "FiLM", 7)]["r2"]:6.2f}'
         f'  ({time.time() - t0:.0f} s)')
if _n_bad:
    raise RuntimeError(f'{_n_bad} winter R² values differ from Section 5: check the code')
logs('all snapshots done; winter R² identical to Section 5 where available')

# ------------------------------------------------------------
# 4. Tables and figures
# ------------------------------------------------------------
E = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(os.path.join(OUT_DIR, 'eval_*_t*.csv')))],
              ignore_index=True)
E.to_csv(os.path.join(OUT_DIR, 'eval_all.csv'), index=False)
E['date'] = pd.to_datetime(E['date'])
E = E.sort_values('date')
T_WIN = pd.Timestamp(f'{pd.Timestamp(START_DATE) + pd.Timedelta(seconds=_T0_S):%Y-%m-%d %H:%M}')
T_SUM = pd.Timestamp(f'{SUMMER_DATE:%Y-%m-%d %H:%M}')
K_SHOW = [2, 7, 12, 17]

# median over the summer collapse window (29 May - 23 Oct 2011) and over the other snapshots
_w = (E.date >= '2011-05-29') & (E.date <= '2011-10-23 23:59')
print('\nMedians: summer window (29 May - 23 Oct 2011) | other snapshots')
print(f'{"region":<11} {"training":<8} {"model":<5} {"k":>3}   {"R2":>13} {"r_pred":>13} {"std_ratio":>13}')
for reg in REG_T:
    for tr in BUNDLES:
        for model in ('LR', 'FiLM'):
            for k in K_SHOW:
                s = E[(E.region == reg) & (E.training == tr) & (E.model == model) & (E.k == k)]
                a, b = s[_w.loc[s.index]], s[~_w.loc[s.index]]
                print(f'{reg:<11} {tr:<8} {model:<5} {k:>3}   '
                      + '   '.join(f'{a[c].median():5.2f} | {b[c].median():5.2f}' for c in ('r2', 'r_pred', 'std_ratio')))

plt.rcParams.update({'font.size': 13, 'axes.titlesize': 13, 'axes.labelsize': 13,
                     'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11,
                     'mathtext.fontset': 'cm'})
C_LR, C_FILM = '#C2410C', '#1D4ED8'
STY = {('winter', 'LR'): dict(color=C_LR, marker='s', ls='-'),
       ('winter', 'FiLM'): dict(color=C_FILM, marker='o', ls='-'),
       ('summer', 'LR'): dict(color=C_LR, marker='s', ls='--', mfc='white'),
       ('summer', 'FiLM'): dict(color=C_FILM, marker='o', ls='--', mfc='white')}
LAB = {'winter': 'trained 3 Feb 2011', 'summer': f'trained {T_SUM:%-d %b %Y}'}


def _panel_figure(col, ylabel, fname, hline):
    fig, axs = plt.subplots(len(REG_T), len(K_SHOW), figsize=(4.6 * len(K_SHOW), 3.8 * len(REG_T)),
                            sharex=True, squeeze=False, constrained_layout=True)
    for a, reg in enumerate(REG_T):
        for b, k in enumerate(K_SHOW):
            ax = axs[a, b]
            for (tr, model), st in STY.items():
                s = E[(E.region == reg) & (E.training == tr) & (E.model == model) & (E.k == k)]
                ax.plot(s.date, s[col], ms=3.5, lw=1.3, label=f'{model}, {LAB[tr]}', **st)
            ax.axhline(hline, color='0.5', lw=0.8)
            for tt in (T_WIN, T_SUM):
                ax.axvline(tt, color='k', ls=':', lw=1.1)
            ax.set_title(f'{"Transition region" if reg == "Transition" else reg}, $k={k}$ ($z={Z_LEVELS[k]:.1f}$ m)')
            ax.grid(alpha=0.3)
            ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
            if b == 0:
                ax.set_ylabel(ylabel)
    axs[0, 0].legend(loc='lower left', frameon=False)
    plt.savefig(os.path.join(OUT_DIR, fname + '.png'), dpi=150)
    plt.savefig(os.path.join(OUT_DIR, fname + '.pdf'))
    plt.show()


_panel_figure('r2', r'$R^2$', 'temporal_r2_winter_summer', 0.0)
_panel_figure('std_ratio', r'$\sigma_{\hat w}\,/\,\sigma_w$', 'temporal_amplitude', 1.0)
logs(f'saved in {OUT_DIR}: eval_<file>_tXX.csv, eval_all.csv, '
     f'temporal_r2_winter_summer.png/pdf, temporal_amplitude.png/pdf')
